# Block 1 — records, ground truth and entity roles

**Reads:** `data/train/train_source{1,2,3}.tsv`, `data/train/train_ground_truth.tsv`,
`data/test/test_source{1,2,3}.tsv` — read-only, supplied by the grader.
**Writes:** `workspace/block1/<B1_VERSION>/` (`B1_VERSION` comes from `er_eval`, so every later block
reads exactly what this notebook wrote).

**Why the language work lives here.** 6.96% of true pairs are cross-script (e.g. S1 in Devanagari,
match in Latin). Character TF-IDF retrieves only **1.82%** of them; romanizing with `anyascii` lifts
that to **58.33%**. The rest is mostly vowel / spelling drift ("Shree" vs "sri", "Traders" vs
"tredrs"), so this notebook also emits a **phonetic skeleton** key that folds both spellings to the
same consonant pattern (`sr blj trdrs`).

### Layers
| Layer | Purpose | Writes |
|---|---|---|
| A / A2 | paths, config, dependencies, I/O helpers | — |
| B | **editable vocabulary** (legal suffixes incl. French, abbreviation maps, junk values, postal rules, country aliases) + record schema | — |
| C / C2 | normalization library + golden tests | — |
| D | country audit + TSV parser row-count check | `vocab/country_map.json` |
| E | ground-truth alignment & integrity (multi-claims merged into one entity group) | `labels/gt_integrity.json` |
| F | entity profiling for stratification | — |
| G | entity-level role assignment (stratified, seeded) | `labels/entity_roles.parquet` |
| H | full record preparation — chunked, multi-core, **resumable** | `records/*/part-*.parquet` |
| I | positive-pair label table | `labels/positive_pairs.parquet` |
| J | QA & leakage checks | `reports/block1_qa_report.json` |
| K | Block 2 readiness bake-off (CE-DEV only) | `reports/block2_key_bakeoff.json` |
| L | manifest + interface | `manifest.json` |

### Output contract (`workspace/block1/<B1_VERSION>/`)
- `records/{train,test}_{s1,s2,s3}/part-*.parquet` — one row per input record, schema in Layer B (`RECORD_SCHEMA`).
- `labels/positive_pairs.parquet` — **authoritative labels**: `(source1_entity_id, matched_entity_id)` + role + cross-script flag.
- `labels/entity_roles.parquet` — per S1 entity: role, stratum, `n_matches`, `is_singleton`, …
- `cluster_id` on every train record = the S1 entity it belongs to (empty for unmatched S2/S3); the
  label of a candidate pair `(s1, x)` is `cluster_id[x] == s1` (exact whenever there are no
  multi-claims — QA checks this).
- `entity_role`: `CE-FIT | CE-DEV | TREE-FIT | TREE-DEV | CALIBRATION | HOLDOUT` for train S1 and
  their matched S2/S3; `POOL` for unmatched train S2/S3 (distractors usable by any role); `TEST` for
  all test records.

### Rules this notebook enforces
1. Nothing is overwritten — raw, native-script normalized, and romanized text are separate columns.
2. Country is an open set — unknown labels pass through normalized, never dropped.
3. No design decision uses CALIBRATION or HOLDOUT — the Layer K bake-off runs on CE-DEV only.
4. No external lookups — transliteration is a local character table (`anyascii`, ISC licence).

**Run order:** top to bottom. Layer H is the long one (tens of minutes on the full data) and resumes
from the last finished shard, so re-running from the top after an interruption is safe. CPU.

## Setup (identical first code cell in every notebook)

`BASE` is the folder that holds `src/` — found by walking up from the working directory, or taken
from `$BER_HOME`. Everything then follows: `DATA = BASE/data` (read-only, supplied by the grader),
`WORKSPACE = BASE/workspace` (everything this pipeline writes). `$BER_WORKSPACE` is set to `WORKSPACE`
**before any module is imported**, because `b2_common` / `b2_prod` compute their paths at import
time. `$HF_HOME` keeps model downloads inside the workspace. Nothing outside `BASE` is read or
written except temporary scratch files.

In [ ]:
# Setup — repository layout, workspace, module path
import os
import sys
import tempfile
from pathlib import Path


def find_base(start=None):
    """The repository folder that holds src/. Override with $BER_HOME."""
    if os.environ.get("BER_HOME"):
        return Path(os.environ["BER_HOME"]).expanduser().resolve()
    here = Path(start or os.getcwd()).resolve()
    for d in (here, *here.parents):
        if (d / "src").is_dir():
            return d
    raise FileNotFoundError(f"no folder containing src/ at or above {here}; set BER_HOME")


BASE = find_base()
SRC, DATA, WORKSPACE = BASE / "src", BASE / "data", BASE / "workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)
ROOT = str(WORKSPACE)

os.environ["BER_WORKSPACE"] = ROOT                       # b2_common / b2_prod read this at import time
os.environ["HF_HOME"] = str(WORKSPACE / "hf_cache")   # model downloads stay inside the workspace
WORK = tempfile.gettempdir()                          # scratch for large local-first writes
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("BASE     :", BASE)
print("DATA     :", DATA, "" if DATA.is_dir() else "(MISSING — supplied by the grader)")
print("WORKSPACE:", WORKSPACE)

import csv
import hashlib
import json
import random
import re
import shutil
import time
import unicodedata
from datetime import datetime

import er_eval as ev

TRAIN_ROOT = DATA / "train"
TEST_ROOT = DATA / "test"

CONFIG = {
    "seed": 42,
    "separator": "\t",
    "profile_sample_rows": 20_000,
    "max_block_size": 5_000,
    "max_candidates_per_source1": 100,
    "pair_sample_limit": 200_000,
}

BLOCK1 = {
    "version": ev.B1_VERSION,                     # the one place this is defined
    "chunk_size": 250_000,                        # rows per shard
    "n_workers": max(1, min(4, os.cpu_count() or 1)),
    "overwrite": False,                           # True = wipe this version's outputs and rebuild
    "tsv_quoting": csv.QUOTE_MINIMAL,             # switch to csv.QUOTE_NONE if Layer D flags a row-count mismatch
    "mixed_script_min_chars": 2,
    "roles": {                                    # entity-level roles (must sum to 1)
        "CE-FIT": 0.30,
        "CE-DEV": 0.05,
        "TREE-FIT": 0.30,
        "TREE-DEV": 0.10,
        "CALIBRATION": 0.10,
        "HOLDOUT": 0.15,
    },
    "match_bucket_edges": [0, 1, 2, 3, 6],        # -> buckets 0 | 1 | 2 | 3-5 | 6+
    "bakeoff_role": "CE-DEV",
    "bakeoff_queries": 2_000,
    "bakeoff_pool_cap": 20_000,                   # candidate pool per country in the bake-off
    "bakeoff_k": CONFIG["max_candidates_per_source1"],
}
assert abs(sum(BLOCK1["roles"].values()) - 1.0) < 1e-9, "role proportions must sum to 1"
ROLE_NAMES = list(BLOCK1["roles"])

BLOCK1_ROOT = WORKSPACE / "block1" / BLOCK1["version"]
RECORDS_ROOT = BLOCK1_ROOT / "records"
LABELS_ROOT = BLOCK1_ROOT / "labels"
VOCAB_ROOT = BLOCK1_ROOT / "vocab"
B1_REPORT_ROOT = BLOCK1_ROOT / "reports"
for path in [RECORDS_ROOT, LABELS_ROOT, VOCAB_ROOT, B1_REPORT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

INPUT_FILES = {
    ("train", "S1"): TRAIN_ROOT / "train_source1.tsv",
    ("train", "S2"): TRAIN_ROOT / "train_source2.tsv",
    ("train", "S3"): TRAIN_ROOT / "train_source3.tsv",
    ("test", "S1"): TEST_ROOT / "test_source1.tsv",
    ("test", "S2"): TEST_ROOT / "test_source2.tsv",
    ("test", "S3"): TEST_ROOT / "test_source3.tsv",
}
GT_FILE = TRAIN_ROOT / "train_ground_truth.tsv"
SOURCE_COLUMNS = ["entity_id", "business_name", "business_address", "country"]

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
random.seed(CONFIG["seed"])
TIMINGS = {}
print("Block 1 output:", BLOCK1_ROOT)
print("Run ID:", RUN_ID, "| workers:", BLOCK1["n_workers"])

## Layer A2 — dependencies, versions, input check, I/O helpers

In [ ]:
# Layer A2: dependencies, versions, input check, I/O helpers

import importlib.metadata as _md

import numpy as np
import pandas as pd
import pyarrow as pa
try:  # pandas >= 3 defaults to Arrow strings whose regex engine treats \w / \b as ASCII-only; keep Python semantics
    pd.set_option("future.infer_string", False)
except Exception:
    pass
try:  # pandas 2.x warns ("silent downcasting") every time .fillna("") lands on an all-string object column;
    # opting into the future behavior now avoids the noise without changing any result on either pandas 2 or 3
    pd.set_option("future.no_silent_downcasting", True)
except Exception:
    pass
import pyarrow.parquet as pq
from anyascii import anyascii as _anyascii

def _version(pkg):
    try:
        return _md.version(pkg)
    except Exception:
        return "unknown"

VERSIONS = {"python": sys.version.split()[0], **{p: _version(p) for p in
            ["pandas", "numpy", "pyarrow", "anyascii", "scikit-learn"]}}
print(VERSIONS)

_missing = [str(p) for p in list(INPUT_FILES.values()) + [GT_FILE] if not p.exists()]
if _missing:
    _found = sorted(str(p.relative_to(DATA)) for p in DATA.rglob("*") if p.is_file()) if DATA.is_dir() else []
    raise FileNotFoundError("Missing input files:\n  " + "\n  ".join(_missing) +
                            f"\nFiles actually present under {DATA}: {_found}")

READ_KW = dict(sep=CONFIG["separator"], dtype=str, keep_default_na=False, na_values=[],
               quoting=BLOCK1["tsv_quoting"])

def iter_tsv(path, usecols=None, chunksize=None):
    """Chunked TSV reader — every raw read in this notebook goes through here."""
    return pd.read_csv(path, usecols=usecols, chunksize=chunksize or BLOCK1["chunk_size"], **READ_KW)

def read_tsv_column(path, cols, chunksize=1_000_000):
    return pd.concat(list(iter_tsv(path, usecols=cols, chunksize=chunksize)), ignore_index=True)

def count_data_lines(path, block=1 << 24):
    """Newline count minus header — used to detect TSV parsing problems (quotes merging rows)."""
    n, last = 0, b""
    with open(path, "rb") as f:
        while True:
            buf = f.read(block)
            if not buf:
                break
            n += buf.count(b"\n")
            last = buf[-1:]
    if last and last != b"\n":
        n += 1
    return n - 1

def ds_key(split, source):
    return f"{split}_{source.lower()}"

def save_json(obj, path):
    path = Path(path)
    tmp = path.with_name("." + path.name + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=str, ensure_ascii=False)
    os.replace(tmp, path)

print("Inputs OK.")

## Layer B — vocabulary & record schema (the one cell to edit)

Everything language- or country-specific lives here so EDA findings (e.g. the add-on's
per-country last-token tables) can be folded in without touching the functions.
Changing anything here changes `CONFIG_HASH`, and Layer H refuses to mix shards built with
different vocabularies (bump `BLOCK1["version"]` or set `overwrite=True`).

Canonical forms are the **short** forms (`limited → ltd`, `road → rd`). Direction doesn't matter
for matching — consistency does.

In [ ]:
# Layer B: vocabulary & record schema

# Normalized values that mean "no value" (compared after casefold + punctuation removal).
JUNK_NORM_VALUES = {"", "na", "n a", "nan", "null", "none", "nil", "not available", "not applicable",
                    "unknown", "xxx", "0", "test"}

# Name prefixes that carry no identity ("M/s." = Messrs, very common in Indian business names).
NAME_PREFIX_PATTERN = r"^\s*(?:m\s*/\s*s|messrs)\b\.?\s*"

# Name token canonicalization (applied to the romanized name; multi-word keys allowed).
NAME_CANON = {
    "private limited": "pvt ltd", "private ltd": "pvt ltd", "pvt limited": "pvt ltd", "pvtltd": "pvt ltd",
    "private": "pvt", "limited": "ltd", "incorporated": "inc", "corporation": "corp", "company": "co",
    "limited liability company": "llc", "limited liability partnership": "llp",
    "public limited company": "plc", "and": "&",
    "brothers": "bros", "international": "intl", "technologies": "tech", "technology": "tech",
    "enterprise": "enterprises", "centre": "center", "service": "services", "industry": "industries",
    # Indic honorific spelling variants — a large source of cross-script drift
    "shree": "sri", "shri": "sri", "sree": "sri", "shreee": "sri", "shrii": "sri",
    "saint": "st",
    # normalizer revision 5 -- anyascii's post-romanization output for these is essentially a
    # SINGLE, fixed spelling across every non-Latin script tested (found via GT cluster_id
    # pairing against each entity's own latin-script record: devanagari/gujarati/bengali/
    # kannada/oriya/telugu). Unlike revision 3/4, this doesn't need per-script native-vocab
    # substitution -- these are anyascii's OWN output, already fully romanized, just not yet
    # in the canonicalization vocabulary that runs before legal-suffix stripping.
    "elelpi": "llp",       # phonetic spell-out of "L-L-P" ("el-el-pi"); ~5.6-20.7% of GT-paired
                            # rows per script had a legal_suffix miss, and this single string
                            # explains nearly all of it across all six scripts.
    "pra li": "pvt ltd",   # abbreviated "Pra(ivate)/Li(mited)", same pattern already handled
                            # for gurmukhi's ਪ੍ਰਾ/ਲਿ -- dominant in devanagari (90,530 occurrences
                            # in the diagnostic sample) and gujarati (10,374), explaining most of
                            # why those two scripts had a larger (~20%) suffix-detection gap.
}

# Address token canonicalization.
ADDR_CANON = {
    "road": "rd", "street": "st", "avenue": "ave", "av": "ave", "boulevard": "blvd", "lane": "ln",
    "drive": "dr", "court": "ct", "place": "pl", "highway": "hwy", "parkway": "pkwy", "suite": "ste",
    "apartment": "apt", "apartments": "apt", "building": "bldg", "floor": "flr", "sector": "sec",
    "number": "no", "near": "nr", "opposite": "opp", "behind": "bhd", "junction": "jn",
    "north": "n", "south": "s", "east": "e", "west": "w", "saint": "st", "mount": "mt",
    "post office": "po", "district": "dist", "chauk": "chowk", "ngr": "nagar",
    "first": "1st", "second": "2nd", "third": "3rd", "fourth": "4th", "fifth": "5th",
}

# Native-script business-vocabulary / honorific canonicalization -- normalizer revision 3.
# anyascii's phonetic transliteration was found (full-scale EDA follow-up diagnostic) to
# garble not just the legal suffix but nearly the WHOLE name for TAMIL, MALAYALAM and
# GURMUKHI specifically: name_latin contained a recognizable business word only 0.64% /
# 0.73% / 0.83% of the time for those three scripts, vs 71.9%-85.9% for every other script
# tested (latin, devanagari, gujarati, bengali, telugu, oriya, kannada). Every entry below
# is a literal native-script token paired with the English spelling it was actually observed
# with for the SAME real-world entity, recovered via a shared cluster_id in the training
# ground truth (181,341 verified cross-script name pairs across 34,011 clusters) -- not a
# guessed transliteration. A handful of tokens where raw co-occurrence favored "ltd"/"pvt"
# purely because those words appear in almost every business name (frequency-absorption
# bias, not true correlation) were corrected by hand using cross-script consensus with an
# already-unambiguous match in a different script for the same word (see conversation notes
# / build_native_canon.py). Applied to name_norm BEFORE anyascii runs (see
# canonicalize_native_vocab / latin_name), so the correct spelling reaches name_latin
# directly instead of anyascii's unrecoverable phonetic noise. Address-vocabulary coverage
# (revision 4): a follow-up exhaustive scan of addr_norm for embedded tamil/malayalam/gurmukhi
# characters (not just addresses where one of these is the DOMINANT script) found 248,552
# addresses -- over 1% of the whole dataset -- carry exactly one native-script word: the
# state name (Tamil Nadu / Kerala / Punjab written in the local script, e.g. "... Chennai
# தமிழ்நாடு"), with the rest of the address already plain Latin/English. anyascii turns
# these into unrecoverable noise ("தமிழ்நாடு" -> "tmilnatu", "ਪੰਜਾਬ" -> "pmjab", "കേരളം" ->
# "kerlm") exactly like the name-side problem, just far narrower in scope: one fixed,
# unambiguous token per script rather than a whole vocabulary, so no GT-pairing exercise was
# needed for these three -- they're standard state names, added directly.
NATIVE_TOKEN_CANON = {
    "tamil": {
        'தமிழ்நாடு': 'tamil nadu',
        'லிமிடெட்': 'ltd',
        'பிரைவேட்': 'pvt',
        'எல்எல்பி': 'llp',
        'குளோபல்': 'global',
        'டெக்': 'tech',
        'இன்டர்நேஷனல்': 'intl',
        'எண்டர்பிரைசஸ்': 'enterprises',
        'ஜெய்': 'jai',
        'ஃபவுண்டேஷன்': 'foundation',
        'ஃபைனான்ஸ்': 'finance',
        'கன்சல்டிங்': 'consulting',
        'ஃபுட்ஸ்': 'foods',
        'கன்ஸ்ட்ரக்ஷன்ஸ்': 'constructions',
        'ஸ்ரீ': 'sri',
        'டெவலப்பர்ஸ்': 'developers',
        'எக்ஸ்போர்ட்ஸ்': 'exports',
        'இன்ஜினியரிங்': 'engineering',
        'இன்வெஸ்ட்மெண்ட்ஸ்': 'investments',
        'கன்சல்டன்ட்ஸ்': 'consultants',
        'ஃபுட்': 'food',
        'அக்ரோ': 'agro',
        'இன்ஃப்ராஸ்ட்ரக்சர்': 'infrastructure',
        'புராடக்ட்ஸ்': 'products',
        'ஹாஸ்பிடாலிட்டி': 'hospitality',
        'வென்ச்சர்ஸ்': 'ventures',
        'இன்ஃபோடெக்': 'infotech',
        'லாஜிஸ்டிக்ஸ்': 'logistics',
        'டிரேடிங்': 'trading',
        'பிசினஸ்': 'business',
        'மேனேஜ்மெண்ட்': 'management',
        'இண்டஸ்ட்ரீஸ்': 'industries',
        'கன்சல்டன்சி': 'consultancy',
        'சிஸ்டம்ஸ்': 'systems',
        'புராஜெக்ட்ஸ்': 'projects',
        'கேர்': 'care',
        'மார்க்கெட்டிங்': 'marketing',
        'ஐடி': 'it',
        'சர்வீசஸ்': 'services',
        'மீடியா': 'media',
        'புரொட்யூசர்': 'producer',
        'இம்பெக்ஸ்': 'impex',
        'இன்வெஸ்ட்மெண்ட்': 'investment',
        'இன்ஃப்ரா': 'infra',
        'எஸ்டேட்': 'estate',
        'சொல்யூஷன்ஸ்': 'solutions',
        'பவர்': 'power',
        'பில்டர்ஸ்': 'builders',
        'எனர்ஜி': 'energy',
        'ஹெல்த்கேர்': 'healthcare',
        'கன்ஸ்ட்ரக்ஷன்': 'construction',
        'டெக்னாலஜீஸ்': 'tech',
        'சாஃப்ட்வேர்': 'software',
        'புராப்பர்ட்டீஸ்': 'properties',
        'டெக்னாலஜி': 'tech',
        'லக்ஷ்மி': 'laxmi',
        'டிரீம்': 'dream',
        'ஈஸ்ட்': 'east',
        'பிரீமியர்': 'premier',
        'ஸ்மார்ட்': 'smart',
        'யுனிக்': 'unique',
        'பயனியர்': 'pioneer',
        'வொயிட்': 'white',
        'ரியல்': 'real',
        'சிவா': 'shiva',
        'சூப்பர்': 'super',
        'சாய்': 'sai',
        'கோல்டு': 'gold',
        'கிருஷ்ணா': 'krishna',
        'சில்வர்': 'silver',
        'ஃபர்ஸ்ட்': 'first',
        'சக்தி': 'shakti',
        'லோட்டஸ்': 'lotus',
        'ஆல்': 'all',
        'பாலாஜி': 'balaji',
        'ராம்': 'ram',
        'குஜராத்': 'gujarat',
        'சதர்ன்': 'southern',
        'ஷ்யாம்': 'shyam',
        'சன்ரைஸ்': 'sunrise',
        'ஹை': 'high',
        'கேலக்ஸி': 'galaxy',
        'லைஃப்': 'life',
        'சன்': 'sun',
        'ஒன்': 'one',
        'டைனமிக்': 'dynamic',
        'இந்தியன்': 'indian',
        'ஜைன்': 'jain',
        'விஜய்': 'vijay',
        'ஃபியூச்சர்': 'future',
        'எஸ்எஸ்': 'ss',
        'ஆல்ஃபா': 'alpha',
        'பிரைம்': 'prime',
        'ஏஸ்': 'ace',
        'சவுத்': 'south',
        'செவன்': 'seven',
        'சிவம்': 'shivam',
        'இண்டோ': 'indo',
        'ஸ்கை': 'sky',
        'யுனிவர்சல்': 'universal',
        'பம்பாய்': 'bombay',
        'நியூ': 'new',
        'ரெட்': 'red',
        'ராஜ்': 'raj',
        'புளூ': 'blue',
        'விஷன்': 'vision',
        'சுப்ரீம்': 'supreme',
        'ஆதித்யா': 'aditya',
        'கிளாசிக்': 'classic',
        'ஸ்டார்': 'star',
        'அபெக்ஸ்': 'apex',
        'ஹரி': 'hari',
        'கிரேட்': 'great',
        'ஃபார்ச்சூன்': 'fortune',
        'பர்ஃபெக்ட்': 'perfect',
        'சிட்டி': 'city',
        'குட்': 'good',
        'மாடர்ன்': 'modern',
        'பிரைட்': 'bright',
        'அர்பன்': 'urban',
        'ஹைடெக்': 'hitech',
        'அரிஹந்த்': 'arihant',
        'ஓம்': 'om',
        'நார்த்': 'north',
        'மை': 'my',
        'பாபா': 'baba',
        'பிக்': 'big',
        'சிவ்': 'shiv',
        'சூர்யா': 'surya',
        'டிஜிட்டல்': 'digital',
        'கிரீன்': 'green',
        'மா': 'maa',
        'திருப்பதி': 'tirupati',
        'யுனைடெட்': 'united',
        'கோல்டன்': 'golden',
        'பிளாக்': 'black',
        'ஹோட்டல்': 'hotel',
        'குரு': 'guru',
        'ராயல்': 'royal',
        'கிரியேட்டிவ்': 'creative',
        'பாரத்': 'bharat',
        'ஆனந்த்': 'anand',
        'ஈஸ்டர்ன்': 'eastern',
        'அல்': 'al',
        'ஸ்வஸ்திக்': 'swastik',
        'பெஸ்ட்': 'best',
        'இன்னோவேட்டிவ்': 'innovative',
        'லட்சுமி': 'lakshmi',
    },
    "malayalam": {
        'കേരളം': 'kerala',
        'ലിമിറ്റഡ്': 'ltd',
        'പ്രൈവറ്റ്': 'pvt',
        'എൽഎൽപി': 'llp',
        'ടെക്': 'tech',
        'ഗ്ലോബൽ': 'global',
        'ഇന്റർനാഷണൽ': 'intl',
        'ശ്രീ': 'sri',
        'കൺസൾട്ടന്റ്സ്': 'consultants',
        'ഇൻഡസ്ട്രീസ്': 'industries',
        'വെഞ്ചേഴ്സ്': 'ventures',
        'ഇൻഫ്ര': 'infra',
        'ബിൽഡേഴ്സ്': 'builders',
        'എന്റർപ്രൈസസ്': 'enterprises',
        'പ്രൊഡക്ട്സ്': 'products',
        'ഇൻഫ്രാസ്ട്രക്ചർ': 'infrastructure',
        'ട്രേഡിംഗ്': 'trading',
        'ഫൗണ്ടേഷൻ': 'foundation',
        'ഹെൽത്ത്കെയർ': 'healthcare',
        'ലക്ഷ്മി': 'lakshmi',
        'കൺസൾട്ടിംഗ്': 'consulting',
        'പ്രോജക്ട്സ്': 'projects',
        'പവർ': 'power',
        'കൺസ്ട്രക്ഷൻ': 'construction',
        'കൺസൾട്ടൻസി': 'consultancy',
        'മാർക്കറ്റിംഗ്': 'marketing',
        'ഹോസ്പിറ്റാലിറ്റി': 'hospitality',
        'എക്സ്പോർട്സ്': 'exports',
        'സർവീസസ്': 'services',
        'ബിസിനസ്': 'business',
        'ടെക്നോളജി': 'tech',
        'ഇൻവെസ്റ്റ്മെന്റ്സ്': 'investments',
        'കെയർ': 'care',
        'പ്രോപ്പർട്ടീസ്': 'properties',
        'പ്രൊഡ്യൂസർ': 'producer',
        'ഡെവലപ്പേഴ്സ്': 'developers',
        'ഫിനാൻസ്': 'finance',
        'ഫുഡ്സ്': 'foods',
        'ഇൻവെസ്റ്റ്മെന്റ്': 'investment',
        'ഇംപെക്സ്': 'impex',
        'സോഫ്റ്റ്വെയർ': 'software',
        'എൻജിനീയറിംഗ്': 'engineering',
        'ഫുഡ്': 'food',
        'കൺസ്ട്രക്ഷൻസ്': 'constructions',
        'ടെക്നോളജീസ്': 'tech',
        'അഗ്രോ': 'agro',
        'ഇൻഫോടെക്': 'infotech',
        'ലോജിസ്റ്റിക്സ്': 'logistics',
        'മാനേജ്മെന്റ്': 'management',
        'സിസ്റ്റംസ്': 'systems',
        'എനർജി': 'energy',
        'ഐടി': 'it',
        'മീഡിയ': 'media',
        'ജയ്': 'jai',
        'എസ്റ്റേറ്റ്': 'estate',
        'സൊല്യൂഷൻസ്': 'solutions',
        'ഈസ്റ്റേൺ': 'eastern',
        'ഡൈനാമിക്': 'dynamic',
        'സിറ്റി': 'city',
        'സ്മാർട്ട്': 'smart',
        'രാജ്': 'raj',
        'വിഷൻ': 'vision',
        'ജൈൻ': 'jain',
        'എസ്എസ്': 'ss',
        'റിയൽ': 'real',
        'ബാബ': 'baba',
        'സെവൻ': 'seven',
        'ഹോട്ടൽ': 'hotel',
        'ഗോൾഡ്': 'gold',
        'പയനിയർ': 'pioneer',
        'ബോംബെ': 'bombay',
        'ബ്ലാക്ക്': 'black',
        'ഫസ്റ്റ്': 'first',
        'ഗുരു': 'guru',
        'ശ്യാം': 'shyam',
        'സൺ': 'sun',
        'ന്യൂ': 'new',
        'ഓൾ': 'all',
        'ലോട്ടസ്': 'lotus',
        'സൺറൈസ്': 'sunrise',
        'ബാലാജി': 'balaji',
        'ബിഗ്': 'big',
        'അപെക്സ്': 'apex',
        'ഇന്ത്യൻ': 'indian',
        'നോർത്ത്': 'north',
        'ആൽഫ': 'alpha',
        'രാം': 'ram',
        'സ്കൈ': 'sky',
        'വൈറ്റ്': 'white',
        'തിരുപ്പതി': 'tirupati',
        'വൺ': 'one',
        'ക്ലാസിക്': 'classic',
        'ശക്തി': 'shakti',
        'വിജയ്': 'vijay',
        'ഗാലക്സി': 'galaxy',
        'യൂണിവേഴ്സൽ': 'universal',
        'മൈ': 'my',
        'സതേൺ': 'southern',
        'സുപ്രീം': 'supreme',
        'ഓം': 'om',
        'ഡ്രീം': 'dream',
        'ഹരി': 'hari',
        'ഗുഡ്': 'good',
        'സ്വസ്തിക്': 'swastik',
        'ബെസ്റ്റ്': 'best',
        'ലൈഫ്': 'life',
        'ക്രിയേറ്റീവ്': 'creative',
        'ആനന്ദ്': 'anand',
        'മാ': 'maa',
        'യുണൈറ്റഡ്': 'united',
        'ഡിജിറ്റൽ': 'digital',
        'റെഡ്': 'red',
        'ഹൈ': 'high',
        'ഗുജറാത്ത്': 'gujarat',
        'മോഡേൺ': 'modern',
        'പ്രൈം': 'prime',
        'ബ്രൈറ്റ്': 'bright',
        'ഹൈടെക്': 'hitech',
        'സൂര്യ': 'surya',
        'സൗത്ത്': 'south',
        'പെർഫെക്റ്റ്': 'perfect',
        'ഫ്യൂച്ചർ': 'future',
        'ഇന്നൊവേറ്റീവ്': 'innovative',
        'ഭാരത്': 'bharat',
        'ഗ്രീൻ': 'green',
        'സ്റ്റാർ': 'star',
        'ശിവം': 'shivam',
        'സിൽവർ': 'silver',
        'ശിവ്': 'shiv',
        'റോയൽ': 'royal',
        'ബ്ലൂ': 'blue',
        'എയ്സ്': 'ace',
        'പ്രീമിയർ': 'premier',
        'അർബൻ': 'urban',
        'ഈസ്റ്റ്': 'east',
        'ആദിത്യ': 'aditya',
        'ഗോൾഡൻ': 'golden',
        'അരിഹന്ത്': 'arihant',
        'യുണീക്ക്': 'unique',
        'അൽ': 'al',
        'ഫോർച്യൂൺ': 'fortune',
        'ഇന്തോ': 'indo',
        'സൂപ്പർ': 'super',
        'ഗ്രേറ്റ്': 'great',
        'സായി': 'sai',
        'കൃഷ്ണ': 'krishna',
        'സായ്': 'sai',
        'ശിവാ': 'shiva',
        'കൃഷ്ണാ': 'krishna',
        'ശിവ': 'shiva',
    },
    "gurmukhi": {
        'ਪੰਜਾਬ': 'punjab',
        'ਲਿਮਟਿਡ': 'ltd',
        'ਪ੍ਰਾਈਵੇਟ': 'pvt',
        'ਪ੍ਰਾ': 'pvt',
        'ਲਿ': 'ltd',
        'ਐਲਐਲਪੀ': 'llp',
        'ਗਲੋਬਲ': 'global',
        'ਟੈਕ': 'tech',
        'ਇੰਟਰਨੈਸ਼ਨਲ': 'intl',
        'ਪ੍ਰੋਜੈਕਟਸ': 'projects',
        'ਕੇਅਰ': 'care',
        'ਪਾਵਰ': 'power',
        'ਸੋਲਿਊਸ਼ਨਜ਼': 'solutions',
        'ਐਂਟਰਪ੍ਰਾਈਜ਼ਿਜ਼': 'enterprises',
        'ਇਸਟੇਟ': 'estate',
        'ਐਗਰੋ': 'agro',
        'ਸਿਸਟਮਜ਼': 'systems',
        'ਪ੍ਰੋਡਕਟਸ': 'products',
        'ਸਰਵਿਸਿਜ਼': 'services',
        'ਫਾਊਂਡੇਸ਼ਨ': 'foundation',
        'ਇੰਜੀਨੀਅਰਿੰਗ': 'engineering',
        'ਇਨਫ੍ਰਾ': 'infra',
        'ਕੰਸਲਟੈਂਟਸ': 'consultants',
        'ਹੈਲਥਕੇਅਰ': 'healthcare',
        'ਫੂਡਜ਼': 'foods',
        'ਟੈਕਨਾਲੋਜੀਜ਼': 'tech',
        'ਕੰਸਟ੍ਰਕਸ਼ਨ': 'construction',
        'ਇਨਵੈਸਟਮੈਂਟਸ': 'investments',
        'ਲਕਸ਼ਮੀ': 'lakshmi',
        'ਇਨਵੈਸਟਮੈਂਟ': 'investment',
        'ਕੰਸਲਟਿੰਗ': 'consulting',
        'ਇੰਡਸਟ੍ਰੀਜ਼': 'industries',
        'ਟੈਕਨਾਲੋਜੀ': 'tech',
        'ਸਾਫਟਵੇਅਰ': 'software',
        'ਹਾਸਪਿਟੈਲਿਟੀ': 'hospitality',
        'ਜੈ': 'jai',
        'ਕੰਸਲਟੈਂਸੀ': 'consultancy',
        'ਲੌਜਿਸਟਿਕਸ': 'logistics',
        'ਟ੍ਰੇਡਿੰਗ': 'trading',
        'ਮੈਨੇਜਮੈਂਟ': 'management',
        'ਪ੍ਰੋਡਿਊਸਰ': 'producer',
        'ਮੀਡੀਆ': 'media',
        'ਇਨਫੋਟੈਕ': 'infotech',
        'ਇੰਪੈਕਸ': 'impex',
        'ਮਾਰਕੀਟਿੰਗ': 'marketing',
        'ਐਕਸਪੋਰਟਸ': 'exports',
        'ਬਿਲਡਰਜ਼': 'builders',
        'ਵੈਂਚਰਜ਼': 'ventures',
        'ਫਾਈਨੈਂਸ': 'finance',
        'ਐਨਰਜੀ': 'energy',
        'ਇਨਫ੍ਰਾਸਟ੍ਰਕਚਰ': 'infrastructure',
        'ਫੂਡ': 'food',
        'ਡਿਵੈਲਪਰਜ਼': 'developers',
        'ਆਈਟੀ': 'it',
        'ਪ੍ਰੋਪਰਟੀਜ਼': 'properties',
        'ਬਿਜ਼ਨਸ': 'business',
        'ਡਿਜੀਟਲ': 'digital',
        'ਫਾਰਚੂਨ': 'fortune',
        'ਡ੍ਰੀਮ': 'dream',
        'ਸਨ': 'sun',
        'ਕੰਸਟ੍ਰਕਸ਼ਨਜ਼': 'constructions',
        'ਸ਼ਿਵਮ': 'shivam',
        'ਸ੍ਰੀ': 'sri',
        'ਮਾਂ': 'maa',
        'ਵਿਜੈ': 'vijay',
        'ਸ਼ਿਆਮ': 'shyam',
        'ਯੂਨੀਕ': 'unique',
        'ਰਾਜ': 'raj',
        'ਫਿਊਚਰ': 'future',
        'ਅਨੰਦ': 'anand',
        'ਕ੍ਰਿਏਟਿਵ': 'creative',
        'ਗੋਲਡ': 'gold',
        'ਏਸ': 'ace',
        'ਹੋਟਲ': 'hotel',
        'ਸੁਪਰੀਮ': 'supreme',
        'ਸਟਾਰ': 'star',
        'ਅਰਿਹੰਤ': 'arihant',
        'ਇਨੋਵੇਟਿਵ': 'innovative',
        'ਰੈੱਡ': 'red',
        'ਡਾਇਨਾਮਿਕ': 'dynamic',
        'ਗੋਲਡਨ': 'golden',
        'ਕ੍ਰਿਸ਼ਨਾ': 'krishna',
        'ਈਸਟ': 'east',
        'ਸ਼ਕਤੀ': 'shakti',
        'ਸੈਵਨ': 'seven',
        'ਬਿਗ': 'big',
        'ਗ੍ਰੇਟ': 'great',
        'ਅਲਫ਼ਾ': 'alpha',
        'ਪ੍ਰੀਮੀਅਰ': 'premier',
        'ਸਵਾਸਤਿਕ': 'swastik',
        'ਯੂਨੀਵਰਸਲ': 'universal',
        'ਸਿਲਵਰ': 'silver',
        'ਗੁਰੂ': 'guru',
        'ਸਕਾਈ': 'sky',
        'ਗੁੱਡ': 'good',
        'ਬੈਸਟ': 'best',
        'ਸਾਊਥ': 'south',
        'ਰਾਮ': 'ram',
        'ਗਲੈਕਸੀ': 'galaxy',
        'ਗ੍ਰੀਨ': 'green',
        'ਸਨਰਾਈਜ਼': 'sunrise',
        'ਹਾਈ': 'high',
        'ਪਾਇਨੀਅਰ': 'pioneer',
        'ਬਾਲਾਜੀ': 'balaji',
        'ਵਿਜ਼ਨ': 'vision',
        'ਸੁਪਰ': 'super',
        'ਬਾਬਾ': 'baba',
        'ਬ੍ਰਾਈਟ': 'bright',
        'ਮਾਡਰਨ': 'modern',
        'ਹਾਈਟੈਕ': 'hitech',
        'ਰੀਅਲ': 'real',
        'ਜੈਨ': 'jain',
        'ਪ੍ਰਾਈਮ': 'prime',
        'ਈਸਟਰਨ': 'eastern',
        'ਗੁਜਰਾਤ': 'gujarat',
        'ਸਦਰਨ': 'southern',
        'ਆਦਿਤਿਆ': 'aditya',
        'ਯੂਨਾਈਟਿਡ': 'united',
        'ਮਾਈ': 'my',
        'ਆਲ': 'all',
        'ਸ਼ਿਵ': 'shiv',
        'ਐੱਸਐੱਸ': 'ss',
        'ਲਾਈਫ਼': 'life',
        'ਵਨ': 'one',
        'ਲੋਟਸ': 'lotus',
        'ਨਿਊ': 'new',
        'ਇੰਡੀਅਨ': 'indian',
        'ਏਪੈਕਸ': 'apex',
        'ਭਾਰਤ': 'bharat',
        'ਸਮਾਰਟ': 'smart',
        'ਤਿਰੂਪਤੀ': 'tirupati',
        'ਇੰਡੋ': 'indo',
        'ਹਰੀ': 'hari',
        'ਬਲੂ': 'blue',
        'ਅਰਬਨ': 'urban',
        'ਅਲ': 'al',
        'ਕਲਾਸਿਕ': 'classic',
        'ਬਲੈਕ': 'black',
        'ਸਿਟੀ': 'city',
        'ਸ਼ਿਵਾ': 'shiva',
        'ਨਾਰਥ': 'north',
        'ਪਰਫੈਕਟ': 'perfect',
        'ਰਾਇਲ': 'royal',
        'ੴ': 'om',
        'ਰੌਇਲ': 'royal',
        'ਬੰਬੇ': 'bombay',
        'ਸਾਈ': 'sai',
        'ਸਾਈਂ': 'sai',
        'ਫਰਸਟ': 'first',
        'ਵਾਈਟ': 'white',
        'ਨੌਰਥ': 'north',
        'ਓਮ': 'om',
        'ਸ਼੍ਰੀ': 'sri',
        'ਵ੍ਹਾਈਟ': 'white',
        'ਸੂਰਜ': 'surya',
        'ਫਸਟ': 'first',
        'ਬੌਂਬੇ': 'bombay',
        'ਸੂਰਿਆ': 'surya',
    },
    # ---- normalizer revision 6 ----
    # The final translation audit (run on the full 24.2M-row real dataset, post revision 5) found
    # two things at far larger scale than anything fixed so far, for the six scripts that never got
    # dictionary treatment (Devanagari, Telugu, Kannada, Gujarati, Bengali, Oriya):
    #   (a) name_norm: anyascii garbles these scripts' OWN native word for "Private" just like it did
    #       for Tamil/Malayalam/Gurmukhi -- e.g. Devanagari "प्राइवेट" -> "praivet", Bengali "প্রাইভেট"
    #       -> "praibhet" -- which strip_legal_suffix's exact-match regex then fails to recognize,
    #       leaving it stuck in name_core. This is not the same failure as revision 5's "elelpi"/"pra
    #       li" (anyascii's own consistent romanized OUTPUT); it's the native-script INPUT never being
    #       canonicalized in the first place, because NATIVE_VOCAB_SCRIPT_RE (below) only ever scanned
    #       for tamil/malayalam/gurmukhi characters. Measured directly on GT-cluster pairs: 46,564
    #       Devanagari / 8,481 Telugu / 7,979 Kannada / 5,394 Gujarati / 6,588 Bengali(as "praibhet") /
    #       1,610 Oriya(as "praibhet") leftover occurrences -- roughly an order of magnitude bigger than
    #       revision 5's "elelpi" count for the same scripts. Fixed the same way as Tamil/Malayalam/
    #       Gurmukhi: canonicalize the real native token (verified directly from the dataset's own
    #       name_norm column, not guessed) before anyascii ever runs.
    #   (b) addr_norm: the Tamil-Nadu/Kerala/Punjab embedded-state-name pattern from revision 4
    #       generalizes to essentially every Indian state, written in its own regional script AND in
    #       Devanagari/Hindi as the de-facto national-language convention, appended to addresses
    #       regardless of that address's own dominant script. Scale: 1,177,483 addresses carry an
    #       embedded Devanagari state name (महाराष्ट्र/दिल्ली/उत्तर प्रदेश/हरियाणा/राजस्थान/...), plus
    #       167,569 Kannada / 150,850 Tamil / 147,487 Telugu / 136,438 Gujarati / 135,012 Bengali /
    #       67,103 Malayalam / 30,503 Gurmukhi / 26,530 Oriya -- over 2,000,000 rows (~8% of the whole
    #       dataset), versus 248,552 for the three-state, three-script version of this bug caught by
    #       revision 4.
    # Both are closed, standard vocabularies (a handful of corporate-suffix words; India's 28 states +
    # 8 union territories) -- no GT-pairing exercise needed, same justification as revision 4's three
    # state entries. The legal-suffix tokens below are the exact native strings the audit's own
    # name_norm scan extracted from the real dataset (not guessed spellings); the state/UT names are
    # standard geography. Devanagari carries the full all-India state/UT list because it is the
    # cross-script convention seen at the largest scale (1.17M rows); the other five scripts get their
    # own state's name (also audit-verified) plus the same closed suffix vocabulary. Because
    # NATIVE_VOCAB_CANON below is a single flat merge of every sub-dict, a token added under any one
    # script becomes available for substitution in a row of ANY dominant script the moment
    # NATIVE_VOCAB_SCRIPT_RE's range covers it -- which is exactly what fixes the cross-script
    # suffix-notation case (e.g. a Devanagari "प्रा. लि." marker on an otherwise-Telugu name) with no
    # extra logic beyond widening that regex to all nine ranges (see below).
    "devanagari": {
        # legal-suffix vocabulary (verified from real name_norm text)
        'प्राइवेट': 'pvt', 'लिमिटेड': 'ltd', 'प्रा': 'pvt', 'लि': 'ltd', 'एलएलपी': 'llp', 'टेक': 'tech',
        # all-India states (Devanagari is the cross-script national-convention script)
        'महाराष्ट्र': 'maharashtra', 'दिल्ली': 'delhi', 'उत्तर': 'uttar', 'प्रदेश': 'pradesh',
        'हरियाणा': 'haryana', 'राजस्थान': 'rajasthan', 'आंध्र': 'andhra', 'अरुणाचल': 'arunachal',
        'असम': 'assam', 'बिहार': 'bihar', 'छत्तीसगढ़': 'chhattisgarh', 'गोवा': 'goa',
        'गुजरात': 'gujarat', 'हिमाचल': 'himachal', 'झारखंड': 'jharkhand', 'झारखण्ड': 'jharkhand',
        'कर्नाटक': 'karnataka', 'केरल': 'kerala', 'मध्य': 'madhya', 'मणिपुर': 'manipur',
        'मेघालय': 'meghalaya', 'मिजोरम': 'mizoram', 'नागालैंड': 'nagaland', 'ओडिशा': 'odisha',
        'उड़ीसा': 'odisha', 'पंजाब': 'punjab', 'सिक्किम': 'sikkim', 'तमिलनाडु': 'tamil nadu',
        'तेलंगाना': 'telangana', 'त्रिपुरा': 'tripura', 'उत्तराखंड': 'uttarakhand',
        'उत्तराखण्ड': 'uttarakhand', 'पश्चिम': 'west', 'बंगाल': 'bengal',
        'चंडीगढ़': 'chandigarh', 'जम्मू': 'jammu', 'कश्मीर': 'kashmir', 'लद्दाख': 'ladakh',
        'लक्षद्वीप': 'lakshadweep', 'पुडुचेरी': 'puducherry', 'पांडिचेरी': 'puducherry',
        'अंडमान': 'andaman', 'निकोबार': 'nicobar', 'दादरा': 'dadra', 'हवेली': 'haveli',
        'दमन': 'daman', 'दीव': 'diu',
    },
    "telugu": {
        'ప్రైవేట్': 'pvt', 'లిమిటెడ్': 'ltd', 'ఎల్ఎల్పీ': 'llp', 'టెక్': 'tech', 'గ్లోబల్': 'global',
        'ఇంటర్నేషనల్': 'intl',
        'తెలంగాణ': 'telangana', 'ఆంధ్రప్రదేశ్': 'andhra pradesh',
    },
    "kannada": {
        'ಪ್ರೈವೇಟ್': 'pvt', 'ಲಿಮಿಟೆಡ್': 'ltd', 'ಎಲ್ಎಲ್ಪಿ': 'llp', 'ಶ್ರೀ': 'sri', 'ಟೆಕ್': 'tech',
        'ಗ್ಲೋಬಲ್': 'global',
        'ಕರ್ನಾಟಕ': 'karnataka',
    },
    "gujarati": {
        'પ્રાઇવેટ': 'pvt', 'લિમિટેડ': 'ltd', 'પ્રા': 'pvt', 'લિ': 'ltd', 'એલએલપી': 'llp',
        'ગ્લોબલ': 'global',
        'ગુજરાત': 'gujarat',
    },
    "bengali": {
        'প্রাইভেট': 'pvt', 'লিমিটেড': 'ltd', 'এলএলপি': 'llp', 'টেক': 'tech', 'গ্লোবাল': 'global',
        'ফুডস': 'foods',
        'পশ্চিমবঙ্গ': 'west bengal',
    },
    "oriya": {
        'ପ୍ରାଇଭେଟ୍': 'pvt', 'ଲିମିଟେଡ୍': 'ltd', 'ଏଲ୍ଏଲ୍ପି': 'llp', 'ମିଡିଆ': 'media',
        'ଗ୍ଲୋବାଲ': 'global', 'ସଲ୍ୟୁସନ୍ସ୍': 'solutions',
        'ଓଡ଼ିଶା': 'odisha',
    },
}
NATIVE_VOCAB_CANON = {tok: en for _d in NATIVE_TOKEN_CANON.values() for tok, en in _d.items()}

# Legal-form suffixes stripped from the END of the romanized name to build name_core.
# "pa" = US professional association: "P.A." -> "p a" -> joined initials -> "pa".
LEGAL_SUFFIXES_GLOBAL = ["pvt ltd", "pvt", "ltd", "llc", "inc", "corp", "co", "llp", "pllc", "plc",
                         "pc", "lp", "pa", "gmbh"]
LEGAL_SUFFIXES_BY_COUNTRY = {           # keys are country_norm values
    "INDIA": ["opc"],
    "FRANCE": ["sarl", "sas", "sasu", "sa", "eurl", "snc", "sci", "scp", "selarl", "sca"],
}

LANDMARK_TERMS = ["nr", "near", "opp", "opposite", "bhd", "behind", "beside", "besides", "next to",
                  "adjacent", "adj", "landmark", "in front of", "infront of", "close to", "above", "below"]

DOMAIN_TLDS = ["co.in", "org.in", "net.in", "firm.in", "gen.in", "ind.in", "co.uk", "com", "net", "org",
               "in", "co", "io", "biz", "info", "us", "fr", "online", "store", "shop", "site", "app",
               "tech", "xyz"]

# ---- Contamination / noise rules (full-scale EDA, add-on Layer O) — normalizer revision 2 ----
# Phone numbers glued onto S2/S3 names ("Nager Coil Services Private - 3306988370") and embedded in some
# addresses ("DOOR NO 8229382409 & 644"). Only CONTIGUOUS runs of >= PHONE_MIN_DIGITS digits count as a
# phone. Shorter or space-separated runs are left alone: they are ZIP+4 codes ("62704 1234") or Oklahoma
# rural addresses ("185389 4235 Road" = house number + road number), not phones.
PHONE_MIN_DIGITS = 10

# Alias markers inside names: "Zetapyra One F/K/A Chattanooga ... LLC", "Zephcira doing business as ...",
# "Vioonyx formerly known as ...". Matched on the raw name with whitespace on both sides. Only the slash
# forms and the spelled-out phrases count: bare or dotted "aka" ("Sri A.K.A. Traders") looks exactly like
# initials, so it is NOT treated as a marker.
NAME_ALIAS_PHRASES = ["doing business as", "formerly known as", "also known as"]
NAME_ALIAS_ABBREVIATIONS = ["d/b/a", "f/k/a", "a/k/a"]

# Leetspeak digit inside a word ("5pecialists", "Mitche1l"). Rewritten only when the token has exactly one
# digit, the digit is in the map, and either (a) it sits between letters and the token has
# >= LEET_MIN_LETTERS letters, or (b) it leads the token and is followed by >= LEET_LEAD_MIN_LETTERS letters.
# So "1st", "3rd", "3m", "b2b", "24x7", "5star", "web3" are left untouched.
LEET_DIGIT_MAP = {"0": "o", "1": "l", "3": "e", "5": "s"}
LEET_MIN_LETTERS = 4
LEET_LEAD_MIN_LETTERS = 6

# Repeated adjacent 1..N-token phrases in names ("Holdings Holdings", "Arvind Enterprises Arvind Enterprises")
# collapse to one copy in name_latin. Applied to every source, so the transform stays symmetric.
NAME_REPEAT_MAX_NGRAM = 3

# addr_is_multi_location: addr_latin token count >= this. The p99 of address word counts is 20-21 in every
# file (full-scale EDA), so this flags roughly the top 1% — the long "Plot No ..., Survey No ..., Village ...,
# Taluk ..., District ..." strings that chain several locations. A length proxy, not an address parser.
ADDR_MULTI_LOCATION_MIN_TOKENS = 22

# Postal-code patterns per country_norm, tried in order on the romanized address (last match wins).
# INDIA: the full-scale EDA found almost no PIN codes in the source addresses (0.0% contain a standalone
# 6-digit run, ~9% contain any 4+ digit run), so postal_code is '' for nearly every India row. That is the
# data, not this rule — downstream blocks must never require postal_code for India.
POSTAL_RULES = {
    "US": [r"^.*\b(\d{5})(?: \d{4})?\b"],
    "INDIA": [r"^.*\b(\d{6})\b", r"^.*\b(\d{3} \d{3})\b"],
    "FRANCE": [r"^.*\b(\d{5})\b", r"^.*\b(\d{2} \d{3})\b"],
}
DEFAULT_POSTAL_RULES = [r"^.*\b(\d{5,6})\b"]   # any country not listed above

# Raw country label (uppercased, non-alphanumerics -> space) -> canonical label. Unlisted labels pass through.
COUNTRY_ALIASES = {
    "USA": "US", "U S A": "US", "U S": "US", "UNITED STATES": "US", "UNITED STATES OF AMERICA": "US",
    "AMERICA": "US", "IN": "INDIA", "IND": "INDIA", "BHARAT": "INDIA", "REPUBLIC OF INDIA": "INDIA",
    "FR": "FRANCE", "FRA": "FRANCE", "REPUBLIQUE FRANCAISE": "FRANCE",
}

# Phonetic skeleton: also fold voiced/unvoiced pairs (g->k, d->t, b->p). Tamil script doesn't mark voicing,
# so "Murugan" vs anyascii "murukn" only agree with this on. Costs some extra collisions — it's a recall key.
SKELETON_FOLD_VOICING = True

SCRIPT_RANGES = [   # (start, end, family) — "latin" includes accented Latin (French)
    (0x0041, 0x005A, "latin"), (0x0061, 0x007A, "latin"), (0x00C0, 0x024F, "latin"), (0x1E00, 0x1EFF, "latin"),
    (0x0370, 0x03FF, "greek"), (0x0400, 0x04FF, "cyrillic"), (0x0600, 0x06FF, "arabic"), (0x0750, 0x077F, "arabic"),
    (0x0900, 0x097F, "devanagari"), (0x0980, 0x09FF, "bengali"), (0x0A00, 0x0A7F, "gurmukhi"),
    (0x0A80, 0x0AFF, "gujarati"), (0x0B00, 0x0B7F, "oriya"), (0x0B80, 0x0BFF, "tamil"),
    (0x0C00, 0x0C7F, "telugu"), (0x0C80, 0x0CFF, "kannada"), (0x0D00, 0x0D7F, "malayalam"),
    (0x0D80, 0x0DFF, "sinhala"), (0x0E00, 0x0E7F, "thai"), (0x3040, 0x30FF, "japanese"),
    (0x4E00, 0x9FFF, "cjk"), (0xAC00, 0xD7AF, "hangul"),
]

# ---- Record schema: the contract every later block reads ----
RECORD_SCHEMA = {
    # column: (dtype, description, main consumers)
    "entity_id": ("string", "original record id (S1-/S2-/S3- prefix)", "all"),
    "source": ("string", "S1 | S2 | S3", "all"),
    "split": ("string", "train | test", "all"),
    "country_raw": ("string", "country label as given", "audit"),
    "country_norm": ("string", "canonical country (open set; unknown labels pass through, empty -> UNKNOWN)", "B2 gate, B3"),
    "name_raw": ("string", "business_name as given", "audit, B4 option"),
    "addr_raw": ("string", "business_address as given", "audit, B4 option"),
    "name_norm": ("string", "NFKC + casefold + punctuation->space; native script KEPT; junk -> ''", "B4 cross-encoder, B3"),
    "addr_norm": ("string", "same as name_norm for the address", "B4 cross-encoder, B3"),
    "name_latin": ("string", "romanized name_norm + canonical tokens + joined initials", "B2 primary key, B3"),
    "addr_latin": ("string", "romanized addr_norm + canonical address tokens", "B2, B3"),
    "name_core": ("string", "name_latin minus legal suffix / leading 'the' / domain TLD (brand anchor)", "B2, B3"),
    "name_skeleton": ("string", "phonetic consonant skeleton of name_core (cross-script recovery key)", "B2 secondary key, B3"),
    "name_aka": ("string", "text after a d/b/a | f/k/a | a/k/a | 'doing business as' | 'formerly known as' | 'also known as' marker, through the name pipeline, legal suffix removed; '' if none. name_latin/name_core keep BOTH sides", "B2 extra key, B3"),
    "legal_suffix": ("string", "canonical legal form removed from the name ('pvt ltd', 'llc', 'sarl', ...)", "B3"),
    "is_domain_name": ("bool", "name is a bare web domain (e.g. xficdt.com)", "B3"),
    "name_script": ("string", "dominant script of name_norm (latin, devanagari, tamil, ..., numeric, empty)", "B3, analysis"),
    "addr_script": ("string", "dominant script of addr_norm", "B3, analysis"),
    "name_is_mixed_script": ("bool", ">=2 scripts with >= mixed_script_min_chars letters", "B3"),
    "name_was_romanized": ("bool", "name_norm was non-ASCII and went through anyascii", "B3, analysis"),
    "addr_was_romanized": ("bool", "addr_norm was non-ASCII and went through anyascii", "B3, analysis"),
    "postal_code": ("string", "country-aware postal code from addr_latin (digits only) or ''; nearly always '' for INDIA (source addresses rarely carry a PIN)", "B2 optional key, B3"),
    "street_number": ("string", "leading / 'no|plot|shop N' street number or ''", "B3"),
    "has_landmark": ("bool", "address uses near/opp/behind-style landmark phrasing", "B3"),
    "name_missing": ("bool", "name empty or placeholder junk", "B3"),
    "addr_missing": ("bool", "address empty or placeholder junk (strong match signal in S2)", "B3"),
    "name_has_embedded_phone": ("bool", "name_raw had a contiguous >= PHONE_MIN_DIGITS digit run; removed before name_norm", "B3, QA"),
    "addr_has_embedded_phone": ("bool", "addr_raw had a contiguous >= PHONE_MIN_DIGITS digit run; removed before addr_norm", "B3, QA"),
    "name_n_tokens": ("int16", "token count of name_latin", "B2 heuristics, B3"),
    "addr_n_tokens": ("int16", "token count of addr_latin", "B2 heuristics, B3"),
    "addr_is_multi_location": ("bool", "addr_n_tokens >= ADDR_MULTI_LOCATION_MIN_TOKENS (length proxy for chained multi-location addresses)", "B3"),
    "cluster_id": ("string", "train: S1 id this record belongs to ('' if unmatched); test: ''", "labels"),
    "entity_role": ("string", "CE-FIT|CE-DEV|TREE-FIT|TREE-DEV|CALIBRATION|HOLDOUT, POOL (unmatched train S2/S3), TEST", "all"),
}
RECORD_COLUMNS = list(RECORD_SCHEMA)

VOCAB = {
    "JUNK_NORM_VALUES": sorted(JUNK_NORM_VALUES), "NAME_PREFIX_PATTERN": NAME_PREFIX_PATTERN,
    "NAME_CANON": NAME_CANON, "ADDR_CANON": ADDR_CANON,
    "LEGAL_SUFFIXES_GLOBAL": LEGAL_SUFFIXES_GLOBAL, "LEGAL_SUFFIXES_BY_COUNTRY": LEGAL_SUFFIXES_BY_COUNTRY,
    "LANDMARK_TERMS": LANDMARK_TERMS, "DOMAIN_TLDS": DOMAIN_TLDS, "POSTAL_RULES": POSTAL_RULES,
    "DEFAULT_POSTAL_RULES": DEFAULT_POSTAL_RULES, "COUNTRY_ALIASES": COUNTRY_ALIASES,
    "SCRIPT_RANGES": SCRIPT_RANGES, "SKELETON_FOLD_VOICING": SKELETON_FOLD_VOICING,
    "PHONE_MIN_DIGITS": PHONE_MIN_DIGITS, "NAME_ALIAS_PHRASES": NAME_ALIAS_PHRASES,
    "NAME_ALIAS_ABBREVIATIONS": NAME_ALIAS_ABBREVIATIONS, "LEET_DIGIT_MAP": LEET_DIGIT_MAP,
    "LEET_MIN_LETTERS": LEET_MIN_LETTERS, "LEET_LEAD_MIN_LETTERS": LEET_LEAD_MIN_LETTERS,
    "NAME_REPEAT_MAX_NGRAM": NAME_REPEAT_MAX_NGRAM, "ADDR_MULTI_LOCATION_MIN_TOKENS": ADDR_MULTI_LOCATION_MIN_TOKENS,
    "NATIVE_VOCAB_CANON": NATIVE_VOCAB_CANON,
}
_hash_payload = {
    "vocab": VOCAB, "schema": RECORD_COLUMNS, "seed": CONFIG["seed"],
    "block1": {k: BLOCK1[k] for k in ["version", "roles", "match_bucket_edges", "mixed_script_min_chars", "tsv_quoting"]},
    "normalizer_revision": 6,   # bump when normalization code in Layer C changes
                                # (2: phone/URL/alias strip, leet, repeats; 3: native-script
                                # tamil/malayalam/gurmukhi NAME vocabulary canonicalization;
                                # 4: same mechanism extended to addr_norm for the 3 embedded
                                # state-name tokens found by the exhaustive address scan;
                                # 5: NAME_CANON gained "elelpi"->"llp" and "pra li"->"pvt ltd",
                                # closing a legal_suffix detection gap on devanagari/bengali/
                                # gujarati/kannada/oriya/telugu -- anyascii's own consistent
                                # romanized output for LLP and abbreviated Private/Limited,
                                # not native-script text, so no per-script dictionary needed;
                                # 6: the final full-scale translation audit (24.2M rows) found the
                                # revision-5 fix only closed the smaller of two gaps. NATIVE_TOKEN_CANON
                                # gained 6 new sub-dicts (devanagari, telugu, kannada, gujarati, bengali,
                                # oriya) covering (a) those scripts' own native word for "Private"/
                                # "Limited"/"Pvt"/"Ltd"/"LLP" -- audit-verified tokens, e.g. Devanagari
                                # "प्राइवेट" which anyascii turns into unrecoverable "praivet" (46,564
                                # occurrences, ~13x bigger than revision 5's "elelpi") -- and (b) the
                                # full all-India state/UT list (Devanagari carries all 36; the other 5
                                # scripts carry their own state), generalizing revision 4's 3-state,
                                # 3-script address fix to what the audit showed is really an all-states,
                                # all-scripts pattern (2M+ rows, ~8% of the dataset, vs 248,552 before).
                                # NATIVE_VOCAB_SCRIPT_RE was widened from 3 Indic ranges to all 9 so
                                # these new entries are actually scanned; canonicalize_native_vocab
                                # itself is unchanged -- coverage is purely a function of the dictionary
                                # + regex range, so this also transparently fixes the cross-script case
                                # (e.g. a Devanagari "प्रा. लि." suffix marker on an otherwise-Telugu
                                # name), since NATIVE_VOCAB_CANON is one flat merge of every sub-dict.
}
CONFIG_HASH = hashlib.sha256(json.dumps(_hash_payload, sort_keys=True, default=str).encode()).hexdigest()[:16]

for _name, _obj in [("name_canon", NAME_CANON), ("addr_canon", ADDR_CANON),
                    ("legal_suffixes", {"global": LEGAL_SUFFIXES_GLOBAL, "by_country": LEGAL_SUFFIXES_BY_COUNTRY}),
                    ("postal_rules", {"by_country": POSTAL_RULES, "default": DEFAULT_POSTAL_RULES}),
                    ("country_aliases", COUNTRY_ALIASES),
                    ("noise_rules", {k: VOCAB[k] for k in [
                        "PHONE_MIN_DIGITS", "NAME_ALIAS_PHRASES", "NAME_ALIAS_ABBREVIATIONS", "LEET_DIGIT_MAP",
                        "LEET_MIN_LETTERS", "LEET_LEAD_MIN_LETTERS", "NAME_REPEAT_MAX_NGRAM",
                        "ADDR_MULTI_LOCATION_MIN_TOKENS"]})]:
    save_json(_obj, VOCAB_ROOT / f"{_name}.json")
print("CONFIG_HASH:", CONFIG_HASH, "| record columns:", len(RECORD_COLUMNS))

## Layer C — normalization library

Pipeline per record (every step is vectorized per chunk; the slow Unicode paths only run on
non-ASCII rows, which are ~10–15% of the data):

```
raw ─► strip ≥10-digit phone runs ─► [name: drop "| www.x.com", split alias on d/b/a | f/k/a | a/k/a | "doing business as" …]
    ─► html-unescape ─► NFKC (non-ASCII only) ─► casefold ─► [name: domain check, strip "M/s"]
    ─► punctuation→space (Unicode-category based, keeps Indic combining marks) ─► junk → ''      = *_norm
*_norm ─► anyascii (non-ASCII only) ─► [name: leetspeak fix 5pecialists → specialists]
       ─► canonical tokens ─► join initials (l l c → llc) ─► [name: collapse repeated phrases]  = *_latin
name_latin ─► strip trailing legal forms / leading "the" / use domain root                      = name_core
alias text ─► same name pipeline, legal form removed                                            = name_aka
name_core ─► phonetic folding (ph→f, sh→s, soft c→s, c→k, anusvara m→n, drop h/vowels,
               optional g/d/b→k/t/p, dedupe letters)                                                = name_skeleton
addr_latin ─► country-aware postal code, street number, landmark flag, multi-location flag
```

Revision 2 (full-scale EDA): the contamination steps run on the **raw** text so a phone number or URL
never reaches `*_norm`, `name_core` or `street_number`; `name_raw` / `addr_raw` keep the original.

Why the punctuation step is category-based: Python's `\w` does **not** match Indic vowel signs
and viramas (Unicode `Mn`/`Mc`), so the usual `re.sub(r"[^\w\s]", " ", s)` silently shreds
Devanagari words into fragments.

In [ ]:
# Layer C: normalization library

import bisect
import functools
import html

WS_RE = re.compile(r"\s+")
ASCII_ALPHA_RE = re.compile(r"[a-z]")
ASCII_DIGIT_RE = re.compile(r"\d")
ALNUM_RE = re.compile(r"[^\W_]")
SINGLE_LETTER_SEQ_RE = re.compile(r"\b(?:[a-z] )+[a-z]\b")
LEADING_ARTICLE_RE = re.compile(r"^the\s+")
EDGE_AMP_RE = re.compile(r"^(?:&\s*)+|(?:\s*&)+$")
NAME_PREFIX_RE = re.compile(NAME_PREFIX_PATTERN)
DOMAIN_ROOT_SEP_RE = re.compile(r"[.\-_]+")
STREET_LEAD_RE = re.compile(r"^(\d{1,6}[a-z]?)\b")
STREET_KEYWORD_RE = re.compile(r"\b(?:no|plot|shop|house|flat|door|unit|bldg|ste|apt|h)\s?(\d{1,6}[a-z]?)\b")
MIXED_MIN = BLOCK1["mixed_script_min_chars"]

# normalizer revision 2
PHONE_RE = re.compile(r"(?:\s*[-–—|:,/]\s*)?(?<!\d)\+?\d{%d,}(?!\d)" % PHONE_MIN_DIGITS)
NAME_URL_RE = re.compile(r"(?<=\w)[\s|,;:\-–—]*(?:https?://|www\.)[^\s|,;]+", re.IGNORECASE)
NAME_ALIAS_RE = re.compile(
    r"\s+(?:" + "|".join([r"\s+".join(re.escape(w) for w in p.split()) for p in NAME_ALIAS_PHRASES]
                          + [r"\s*/\s*".join(re.escape(c) for c in a.split("/")) + r"\b" for a in NAME_ALIAS_ABBREVIATIONS])
    + r")\s+", re.IGNORECASE)
_LEET_DIGITS = "".join(sorted(LEET_DIGIT_MAP))
LEET_CANDIDATE_RE = re.compile(r"[a-z][%s][a-z]|\b[%s][a-z]{%d}" % (_LEET_DIGITS, _LEET_DIGITS, LEET_LEAD_MIN_LETTERS))
REPEAT_NGRAM_RE = re.compile(r"\b(\w+(?: \w+){0,%d})(?: \1\b)+" % (NAME_REPEAT_MAX_NGRAM - 1))
# normalizer revision 3 -- tamil (0B80-0BFF) / malayalam (0D00-0D7F) / gurmukhi (0A00-0A7F) code points.
# normalizer revision 6 -- widened to all 9 major Indic ranges (devanagari 0900-097F, bengali 0980-09FF,
# gurmukhi 0A00-0A7F, gujarati 0A80-0AFF, oriya 0B00-0B7F, tamil 0B80-0BFF, telugu 0C00-0C7F,
# kannada 0C80-0CFF, malayalam 0D00-0D7F) so the six scripts added to NATIVE_TOKEN_CANON above actually
# get scanned -- the dictionary additions alone did nothing until this regex could see their characters.
NATIVE_VOCAB_SCRIPT_RE = re.compile(
    r"[\u0900-\u097f\u0980-\u09ff\u0a00-\u0a7f\u0a80-\u0aff\u0b00-\u0b7f\u0b80-\u0bff"
    r"\u0c00-\u0c7f\u0c80-\u0cff\u0d00-\u0d7f]"
)

def _build_punct_table():
    table = {}
    for cp in range(sys.maxunicode + 1):
        cat = unicodedata.category(chr(cp))
        if cat[0] in ("P", "S") or cat == "Cc":
            table[cp] = " "
        elif cat == "Cf":            # ZWJ / ZWNJ / BOM / soft hyphen: remove without splitting the word
            table[cp] = ""
    for ch in "'’‘`´":           # apostrophes glue ("mcdonald's" -> "mcdonalds")
        table[ord(ch)] = ""
    table[ord("&")] = " & "
    return table

PUNCT_TABLE = _build_punct_table()

def _alternation(terms):
    terms = sorted(set(terms), key=lambda t: (-len(t), t))
    return "|".join(re.escape(t) for t in terms)

NAME_CANON_RE = re.compile(r"\b(" + _alternation(NAME_CANON) + r")\b")
ADDR_CANON_RE = re.compile(r"\b(" + _alternation(ADDR_CANON) + r")\b")
LANDMARK_RE = re.compile(r"\b(?:" + _alternation(LANDMARK_TERMS) + r")\b")
DOMAIN_RE = re.compile(r"^(?:https?://)?(?:www\.)?([a-z0-9][a-z0-9-]*(?:\.[a-z0-9-]+)*?)\.(?:"
                       + _alternation(DOMAIN_TLDS) + r")/?$")

def _suffix_regex(terms):
    return re.compile(r"((?:\s+(?:" + _alternation(terms) + r"))+)$")

SUFFIX_RE_GLOBAL = _suffix_regex(LEGAL_SUFFIXES_GLOBAL)
SUFFIX_RE_BY_COUNTRY = {c: _suffix_regex(LEGAL_SUFFIXES_GLOBAL + extra)
                        for c, extra in LEGAL_SUFFIXES_BY_COUNTRY.items()}
POSTAL_RE_BY_COUNTRY = {c: [re.compile(p) for p in pats] for c, pats in POSTAL_RULES.items()}
DEFAULT_POSTAL_RES = [re.compile(p) for p in DEFAULT_POSTAL_RULES]

_SR = sorted(SCRIPT_RANGES)
_SR_STARTS = [r[0] for r in _SR]

# ---------------------------------------------------------------- scripts
def _cp_family(cp):
    i = bisect.bisect_right(_SR_STARTS, cp) - 1
    if i >= 0 and cp <= _SR[i][1]:
        return _SR[i][2]
    return "other"

def _script_of_nonascii(text):
    counts = {}
    for ch in text:
        if ch.isalpha():
            fam = _cp_family(ord(ch))
            counts[fam] = counts.get(fam, 0) + 1
    if not counts:
        return ("numeric" if any(ch.isdigit() for ch in text) else "empty"), False
    dominant = max(sorted(counts), key=counts.get)
    mixed = sum(1 for v in counts.values() if v >= MIXED_MIN) > 1
    return dominant, mixed

def detect_scripts(norm):
    """Vectorized: returns (dominant_script, is_mixed) Series for normalized text.
    `script` is object-dtype so masked string assignment is safe; `mixed` is bool-dtype, so it is built as a
    numpy array and assigned by position (same reasoning as junk_mask's fix above) rather than via
    `mixed[bool_mask] = bool_array`, which pandas < 3 can silently no-op."""
    is_ascii_arr = norm.map(str.isascii).to_numpy()
    script = pd.Series("empty", index=norm.index, dtype=object)
    mixed_arr = np.zeros(len(norm), dtype=bool)
    idx_ascii = np.flatnonzero(is_ascii_arr)
    idx_native = np.flatnonzero(~is_ascii_arr)
    if idx_ascii.size:
        a = norm.iloc[idx_ascii]
        script.iloc[idx_ascii] = np.where(a.str.contains(ASCII_ALPHA_RE), "latin",
                                          np.where(a.str.contains(ASCII_DIGIT_RE), "numeric", "empty"))
    if idx_native.size:
        res = norm.iloc[idx_native].map(_script_of_nonascii)
        script.iloc[idx_native] = res.map(lambda r: r[0]).to_numpy()
        mixed_arr[idx_native] = res.map(lambda r: r[1]).to_numpy(dtype=bool)
    mixed = pd.Series(mixed_arr, index=norm.index)
    return script, mixed

def script_family(script):
    if script == "latin":
        return "latin"
    if script in ("empty", "numeric"):
        return "none"
    return script

def script_group(script, mixed):
    if script in ("empty", "numeric"):
        return "none"
    if mixed:
        return "mixed"
    return "latin" if script == "latin" else "non_latin"

def cross_script_flags(a, b):
    """True where both sides have letters and their script families differ."""
    fa, fb = a.map(script_family), b.map(script_family)
    return (fa != fb) & (fa != "none") & (fb != "none")

# ---------------------------------------------------------------- text normalization
def pre_normalize(s):
    s = s.fillna("").astype(str)
    has_ent = s.str.contains("&", regex=False) & s.str.contains(";", regex=False)
    if has_ent.any():
        s = s.where(~has_ent, s[has_ent].map(html.unescape))
    nonascii = ~s.map(str.isascii)
    if nonascii.any():
        s = s.where(~nonascii, s[nonascii].map(lambda x: unicodedata.normalize("NFKC", x)))
    return s.str.casefold()

def finish_normalize(s):
    return s.str.translate(PUNCT_TABLE).str.replace(WS_RE, " ", regex=True).str.strip()

def _has_alnum(text):
    return any(ch.isalnum() for ch in text)

def junk_mask(norm):
    """Empty, placeholder, or no letter/digit in any script (Unicode-aware, independent of regex engine).
    Builds the result as a plain numpy bool array and assigns by position, never via `series[bool_mask] = arr`
    into a bool-dtype Series — pandas < 3 can silently drop that assignment (kept as all-False) instead of
    raising, which showed up as name_norm/addr_norm going empty for every row on a mixed-script chunk."""
    is_ascii_arr = norm.map(str.isascii).to_numpy()
    has = np.zeros(len(norm), dtype=bool)
    idx_ascii = np.flatnonzero(is_ascii_arr)
    idx_native = np.flatnonzero(~is_ascii_arr)
    if idx_ascii.size:
        has[idx_ascii] = norm.iloc[idx_ascii].str.contains(r"[a-z0-9]", regex=True).to_numpy()
    if idx_native.size:
        has[idx_native] = norm.iloc[idx_native].map(_has_alnum).to_numpy(dtype=bool)
    has = pd.Series(has, index=norm.index)
    return norm.isin(JUNK_NORM_VALUES) | ~has

def detect_domain(cf):
    root = cf.str.strip().str.extract(DOMAIN_RE, expand=False)
    is_dom = root.notna()
    root = root.fillna("").str.replace(DOMAIN_ROOT_SEP_RE, " ", regex=True).str.strip()
    return is_dom, root

def normalize_name(raw):
    cf = pre_normalize(raw)
    is_domain, domain_root = detect_domain(cf)
    cf = cf.str.replace(NAME_PREFIX_RE, "", regex=True)
    norm = finish_normalize(cf)
    missing = junk_mask(norm)
    return norm.where(~missing, ""), missing, is_domain & ~missing, domain_root

def normalize_address(raw):
    norm = finish_normalize(pre_normalize(raw))
    missing = junk_mask(norm)
    return norm.where(~missing, ""), missing

def romanize(norm):
    nonascii = ~norm.map(str.isascii)
    out = norm
    if nonascii.any():
        rom = (norm[nonascii].map(_anyascii).str.casefold().str.translate(PUNCT_TABLE)
               .str.replace(WS_RE, " ", regex=True).str.strip())
        out = norm.where(~nonascii, rom)
    return out, nonascii

def canonicalize(s, rx, mapping):
    s = s.str.replace(rx, lambda m: mapping[m.group(1)], regex=True)
    s = s.str.replace(SINGLE_LETTER_SEQ_RE, lambda m: m.group(0).replace(" ", ""), regex=True)
    return s.str.replace(WS_RE, " ", regex=True).str.strip()

def canonicalize_native_vocab(norm):
    """Native-script token -> English spelling, BEFORE anyascii runs (normalizer revision 3).
    Scoped to name_norm only. See NATIVE_TOKEN_CANON above for provenance. Whitespace-token
    substitution: only rows containing a tamil/malayalam/gurmukhi character are touched
    (cheap regex pre-filter, same pattern as fix_leet's `cand` check); every other row,
    including every other script, passes through unchanged. Unmatched tokens within a
    touched row (a personal name not in the table, a number, ...) also pass through
    unchanged and still go through anyascii as before -- this only ever adds coverage, it
    never removes anyascii's fallback behavior."""
    cand = norm.str.contains(NATIVE_VOCAB_SCRIPT_RE, regex=True)
    if not cand.any():
        return norm
    sub = norm[cand].map(lambda s: " ".join(NATIVE_VOCAB_CANON.get(t, t) for t in s.split(" ")))
    return norm.where(~cand, sub)

def strip_legal_suffix(name_latin, country):
    core = name_latin.copy()
    suffix = pd.Series("", index=name_latin.index, dtype=object)
    for c, idx in name_latin.groupby(country, sort=False).groups.items():
        rx = SUFFIX_RE_BY_COUNTRY.get(c, SUFFIX_RE_GLOBAL)
        sub = name_latin.loc[idx]
        suffix.loc[idx] = sub.str.extract(rx, expand=False).fillna("").str.replace(WS_RE, " ", regex=True).str.strip()
        core.loc[idx] = sub.str.replace(rx, "", regex=True)
    core = core.str.replace(EDGE_AMP_RE, "", regex=True).str.replace(LEADING_ARTICLE_RE, "", regex=True).str.strip()
    return core, suffix

# ---------------------------------------------------------------- contamination / noise (revision 2)
def strip_phone(raw):
    """Remove contiguous >= PHONE_MIN_DIGITS digit runs (plus a leading ' - ' / ' | ' separator).
    Returns (cleaned_raw, had_phone)."""
    raw = raw.fillna("").astype(str)
    has = raw.str.contains(PHONE_RE)
    if has.any():
        raw = raw.where(~has, raw[has].str.replace(PHONE_RE, " ", regex=True).str.strip())
    return raw, has.astype(bool)

def strip_name_url(raw):
    """Drop a 'www.x.com' / 'http(s)://...' that follows other text ("... LLC | www.woolridge.com").
    A name that IS a domain is left alone for detect_domain."""
    has = raw.str.contains(NAME_URL_RE)
    if has.any():   # DOMAIN_RE has a capture group, so match row-wise on the (rare) candidates
        whole_domain = raw[has].str.strip().str.casefold().map(lambda s: DOMAIN_RE.match(s) is not None)
        has.loc[whole_domain.index[whole_domain]] = False
    if has.any():
        raw = raw.where(~has, raw[has].str.replace(NAME_URL_RE, " ", regex=True).str.strip())
    return raw

def split_alias(raw):
    """'Zephcira doing business as Behavioral Health LLC' ->
    ('Zephcira Behavioral Health LLC', 'Behavioral Health LLC'). The name keeps BOTH sides (the EDA does
    not show which side S1 uses); the text after the marker is returned separately ('' if no marker)."""
    alias = pd.Series("", index=raw.index, dtype=object)
    has = raw.str.contains(NAME_ALIAS_RE)
    if has.any():
        parts = raw[has].str.split(NAME_ALIAS_RE, n=1, regex=True)
        head = parts.str[0].fillna("").str.strip()
        tail = parts.str[1].fillna("").str.strip()
        ok = (head != "") & (tail != "")
        idx = head.index[ok]
        raw = raw.copy()
        raw.loc[idx] = head[ok] + " " + tail[ok]
        alias.loc[idx] = tail[ok]
    return raw, alias

@functools.lru_cache(maxsize=200_000)
def _leet_token(tok):
    pos = [i for i, ch in enumerate(tok) if ch.isdigit()]
    if len(pos) != 1 or not tok.isascii() or not tok.isalnum():
        return tok
    i = pos[0]
    if tok[i] not in LEET_DIGIT_MAP:
        return tok
    n_letters = len(tok) - 1
    if i == 0:
        ok = n_letters >= LEET_LEAD_MIN_LETTERS
    elif i == len(tok) - 1:
        ok = False
    else:
        ok = n_letters >= LEET_MIN_LETTERS
    return tok[:i] + LEET_DIGIT_MAP[tok[i]] + tok[i + 1:] if ok else tok

def fix_leet(latin):
    """'first 5pecialists' -> 'first specialists', 'mitche1l' -> 'mitchell'; '3m', 'b2b', '1st' untouched."""
    cand = latin.str.contains(LEET_CANDIDATE_RE)
    if cand.any():
        latin = latin.where(~cand, latin[cand].map(lambda s: " ".join(_leet_token(t) for t in s.split())))
    return latin

def collapse_repeats(latin):
    """'trust hospitalities india holdings holdings pvt ltd' -> 'trust hospitalities india holdings pvt ltd'."""
    return latin.str.replace(REPEAT_NGRAM_RE, lambda m: m.group(1), regex=True)

def latin_name(norm):
    """name_norm -> name_latin (native-vocab canon, romanize, leet fix, canonical tokens, repeat collapse)."""
    norm = canonicalize_native_vocab(norm)
    latin, rom = romanize(norm)
    return collapse_repeats(canonicalize(fix_leet(latin), NAME_CANON_RE, NAME_CANON)), rom

# ---------------------------------------------------------------- phonetic skeleton
_SK_DIGRAPHS = [("ph", "f"), ("gh", "g"), ("kh", "k"), ("bh", "b"), ("dh", "d"), ("th", "t"), ("jh", "j"),
                ("sh", "s"), ("ch", "c"), ("ck", "k"), ("q", "k"), ("x", "ks"), ("z", "j"), ("w", "v")]
_SK_SOFT_C = re.compile(r"c(?=[eiy])")
_SK_NASAL_M = re.compile(r"m(?=[cdfgjklnqrstvxz])")   # anusvara: anyascii writes it as 'm'
_SK_DROP = re.compile(r"[aeiouyh]")
_SK_REPEAT = re.compile(r"([a-z])\1+")          # letters only — never touch digits
_SK_VOICING = str.maketrans({"g": "k", "d": "t", "b": "p"})

@functools.lru_cache(maxsize=500_000)
def _skeleton_token(tok):
    if tok.isdigit():
        return tok
    t = tok
    for a, b in _SK_DIGRAPHS:
        t = t.replace(a, b)
    t = _SK_SOFT_C.sub("s", t).replace("c", "k")
    t = _SK_NASAL_M.sub("n", t)
    t = _SK_DROP.sub("", t)
    if SKELETON_FOLD_VOICING:
        t = t.translate(_SK_VOICING)
    return _SK_REPEAT.sub(r"\1", t)

def phonetic_skeleton(text):
    """Indic-English phonetic folding: 'shree balaji traders' and anyascii's 'sri balaji tredrs'
    fold to the same key ('sr plj trtrs' with voicing folding on). Lossy on purpose — a recall key,
    never a match decision."""
    out = [_skeleton_token(t) for t in text.split() if t != "&"]
    return " ".join(t for t in out if t)

# ---------------------------------------------------------------- address components / country
def extract_postal(addr_latin, country):
    out = pd.Series("", index=addr_latin.index, dtype=object)
    for c, idx in addr_latin.groupby(country, sort=False).groups.items():
        sub = addr_latin.loc[idx]
        found = pd.Series(np.nan, index=sub.index, dtype=object)
        for rx in POSTAL_RE_BY_COUNTRY.get(c, DEFAULT_POSTAL_RES):
            need = found.isna()
            if not need.any():
                break
            found[need] = sub[need].str.extract(rx, expand=False)
        out.loc[idx] = found.fillna("").str.replace(" ", "", regex=False)
    return out

def extract_street_number(addr_latin, postal):
    lead = addr_latin.str.extract(STREET_LEAD_RE, expand=False)
    kw = addr_latin.str.extract(STREET_KEYWORD_RE, expand=False)
    num = lead.fillna(kw).fillna("")
    return num.where(num != postal, "")

@functools.lru_cache(maxsize=10_000)
def normalize_country(raw):
    key = _anyascii(unicodedata.normalize("NFKC", raw or "")).upper()
    key = re.sub(r"[^A-Z0-9]+", " ", key).strip()
    if not key:
        return "UNKNOWN"
    return COUNTRY_ALIASES.get(key, key)

def n_tokens(s):
    return np.where(s.str.len() > 0, s.str.count(" ") + 1, 0).astype("int16")

# ---------------------------------------------------------------- chunk preparation
def prepare_chunk(df):
    """Raw TSV chunk -> prepared columns (everything except source/split/cluster_id/entity_role)."""
    df = df.reset_index(drop=True)
    out = pd.DataFrame(index=df.index)
    out["entity_id"] = df["entity_id"].fillna("").astype(str).str.strip()
    out["country_raw"] = df["country"].fillna("").astype(str)
    out["country_norm"] = out["country_raw"].map(normalize_country)
    out["name_raw"] = df["business_name"].fillna("").astype(str)
    out["addr_raw"] = df["business_address"].fillna("").astype(str)

    # Contamination is removed from the RAW text first (revision 2); name_raw / addr_raw keep the original.
    name_clean, name_phone = strip_phone(out["name_raw"])
    name_clean = strip_name_url(name_clean)
    name_clean, alias_raw = split_alias(name_clean)
    addr_clean, addr_phone = strip_phone(out["addr_raw"])

    name_norm, name_missing, is_domain, domain_root = normalize_name(name_clean)
    addr_norm, addr_missing = normalize_address(addr_clean)

    name_latin, name_rom = latin_name(name_norm)
    addr_norm_canon = canonicalize_native_vocab(addr_norm)  # revision 4: state-name coverage
    addr_latin, addr_rom = romanize(addr_norm_canon)
    addr_latin = canonicalize(addr_latin, ADDR_CANON_RE, ADDR_CANON)

    core, suffix = strip_legal_suffix(name_latin, out["country_norm"])
    if is_domain.any():
        dom_core = canonicalize(domain_root[is_domain], NAME_CANON_RE, NAME_CANON)
        core = core.where(~is_domain, dom_core)
    core = core.where(core.str.len() > 0, name_latin)

    # name_aka: text after an alias marker, through the same name pipeline, legal suffix removed.
    name_aka = pd.Series("", index=df.index, dtype=object)
    has_alias = alias_raw != ""
    if has_alias.any():
        a_latin, _ = latin_name(normalize_name(alias_raw[has_alias])[0])
        a_core, _ = strip_legal_suffix(a_latin, out["country_norm"][has_alias])
        name_aka.loc[has_alias] = a_core.where(a_core.str.len() > 0, a_latin)

    name_script, name_mixed = detect_scripts(name_norm)
    addr_script, _ = detect_scripts(addr_norm)
    postal = extract_postal(addr_latin, out["country_norm"])

    out["name_norm"] = name_norm
    out["addr_norm"] = addr_norm
    out["name_latin"] = name_latin
    out["addr_latin"] = addr_latin
    out["name_core"] = core
    out["name_skeleton"] = core.map(phonetic_skeleton)
    out["name_aka"] = name_aka
    out["legal_suffix"] = suffix
    out["is_domain_name"] = is_domain.astype(bool)
    out["name_script"] = name_script
    out["addr_script"] = addr_script
    out["name_is_mixed_script"] = name_mixed.astype(bool)
    out["name_was_romanized"] = name_rom.astype(bool)
    out["addr_was_romanized"] = addr_rom.astype(bool)
    out["postal_code"] = postal
    out["street_number"] = extract_street_number(addr_latin, postal)
    out["has_landmark"] = addr_latin.str.contains(LANDMARK_RE).astype(bool)
    out["name_missing"] = name_missing.astype(bool)
    out["addr_missing"] = addr_missing.astype(bool)
    out["name_has_embedded_phone"] = name_phone
    out["addr_has_embedded_phone"] = addr_phone
    out["name_n_tokens"] = n_tokens(name_latin)
    out["addr_n_tokens"] = n_tokens(addr_latin)
    out["addr_is_multi_location"] = (out["addr_n_tokens"] >= ADDR_MULTI_LOCATION_MIN_TOKENS).astype(bool)
    return out

print("Normalization library ready. Punctuation table entries:", len(PUNCT_TABLE))

## Layer C2 — golden tests

Hand-written examples covering the failure modes the EDA surfaced. If any assertion fails, fix
the vocabulary / functions before running Layer H. The last table shows cross-script pairs side
by side — the skeleton key should agree even where `name_latin` doesn't.

In [ ]:
# Layer C2: golden tests

_golden = pd.DataFrame([
    # entity_id, business_name, business_address, country
    ("G-01", "M/s. Shree Balaji Traders Pvt. Ltd.", "Shop No. 5, Near SBI ATM, MG Road, Pune 411 001", "India"),
    ("G-02", "श्री बालाजी ट्रेडर्स", "पुणे ४११००१", "India"),
    ("G-03", "xficdt.com", "", "India"),
    ("G-04", "Boulangerie Du Marché SARL", "12 Rue de l'Église, 75001 Paris", "France"),
    ("G-05", "ACME Corporation, Inc.", "1600 North Main Street, Springfield, IL 62704-1234", "USA"),
    ("G-06", "N/A", "-", "india "),
    ("G-07", "Tom &amp; Jerry LLC", "PO Box 12", "U.S.A."),
    ("G-08", "Smith L.L.C.", "22 W 5th Ave", "US"),
    ("G-09", "गणेश मोटर्स", "", "India"),
    ("G-10", "Ganesh Motors", "", "India"),
    ("G-11", "The Coffee House & Co", "", "Freedonia"),
    # normalizer revision 2 — contamination patterns from the full-scale EDA (add-on Layer O)
    ("G-12", "Johnson Dental P.A.", "185389 4235 Road, Finley, OK", "US"),
    ("G-13", "Nager Coil Services Private - 3306988370",
     "DOOR NO 8229382409 & 644, ROAD NO. 36, SHAIKPET, Telangana", "India"),
    ("G-14", "Zephcira doing business as Behavioral Health Heartland Medicine LLC", "", "US"),
    ("G-15", "Zetapyra One F/K/A Chattanooga Family National Chiropractic LLC", "", "US"),
    ("G-16", "TRUST HOSPITALITIES (INDIA) HOLDINGS HOLDINGS PRIVATE LIMITED", "", "India"),
    ("G-17", "The Physical Therapy First 5pecialists Lakeside LLC - 4168490901", "", "US"),
    ("G-18", "WOOLRIDGE SILVER FRESENIUS GREATER  LLC | www.woolridge.com", "", "US"),
    ("G-19", "Sri A.K.A. Traders", "", "India"),
    ("G-20", "solutionsrrrcoopers.com - 9871317824", "", "India"),
    ("G-21", "3M 24x7 B2B Services", "", "US"),
    ("G-22", "Acme Industries", "Plot No 28, Survey No. 202, Sompura 1St Stage, Kiadb Industrial Area, Dabaspet, "
     "Nidvanda Village, Sompur Hobli, Nelmangla Taluk, Bangalore, Nelamangala, Bangalore Rural, Karnataka", "India"),
    ("G-23", "Mitche1l Premium LLC", "", "US"),
], columns=SOURCE_COLUMNS)

g = prepare_chunk(_golden).set_index("entity_id")

def check(eid, col, expected):
    got = g.loc[eid, col]
    assert got == expected, f"{eid}.{col}: expected {expected!r}, got {got!r}"

check("G-01", "name_latin", "sri balaji traders pvt ltd")
check("G-01", "name_core", "sri balaji traders")
check("G-01", "legal_suffix", "pvt ltd")
check("G-01", "name_skeleton", "sr plj trtrs" if SKELETON_FOLD_VOICING else "sr blj trdrs")
check("G-01", "postal_code", "411001")
check("G-01", "street_number", "5")
assert g.loc["G-01", "has_landmark"]
check("G-02", "name_script", "devanagari")
assert g.loc["G-02", "name_was_romanized"]
check("G-02", "name_skeleton", g.loc["G-01", "name_skeleton"])        # the cross-script win
check("G-02", "postal_code", "411001")                                # Devanagari digits
assert g.loc["G-03", "is_domain_name"]
check("G-03", "name_core", "xficdt")
assert g.loc["G-03", "addr_missing"]
check("G-04", "name_core", "boulangerie du marche")
check("G-04", "legal_suffix", "sarl")
check("G-04", "postal_code", "75001")
check("G-04", "street_number", "12")
check("G-04", "country_norm", "FRANCE")
check("G-05", "name_core", "acme")
check("G-05", "legal_suffix", "corp inc")
check("G-05", "postal_code", "62704")
check("G-05", "street_number", "1600")
check("G-05", "country_norm", "US")
assert g.loc["G-06", "name_missing"] and g.loc["G-06", "addr_missing"]
check("G-06", "name_norm", "")
check("G-06", "country_norm", "INDIA")
check("G-07", "name_core", "tom & jerry")
check("G-07", "legal_suffix", "llc")
check("G-07", "country_norm", "US")
check("G-08", "name_core", "smith")
check("G-08", "legal_suffix", "llc")
check("G-09", "name_skeleton", g.loc["G-10", "name_skeleton"])
check("G-11", "country_norm", "FREEDONIA")                             # open set: passes through
check("G-11", "name_core", "coffee house")

# normalizer revision 2
check("G-12", "name_core", "johnson dental")
check("G-12", "legal_suffix", "pa")
check("G-12", "street_number", "185389")                               # Oklahoma rural address, not a phone
assert not g.loc["G-12", "addr_has_embedded_phone"] and not g.loc["G-05", "addr_has_embedded_phone"]  # ZIP+4 too
assert g.loc["G-13", "name_has_embedded_phone"] and g.loc["G-13", "addr_has_embedded_phone"]
check("G-13", "name_core", "nager coil services")
assert "8229382409" not in g.loc["G-13", "addr_latin"]
check("G-13", "street_number", "36")
check("G-14", "name_aka", "behavioral health heartland medicine")
check("G-14", "name_core", "zephcira behavioral health heartland medicine")    # both sides kept
check("G-15", "name_aka", "chattanooga family national chiropractic")
check("G-15", "legal_suffix", "llc")
check("G-16", "name_latin", "trust hospitalities india holdings pvt ltd")     # stutter collapsed
check("G-17", "name_core", "physical therapy first specialists lakeside")     # phone gone, leet fixed
assert g.loc["G-17", "name_has_embedded_phone"]
check("G-18", "name_core", "woolridge silver fresenius greater")              # URL tail gone
check("G-18", "legal_suffix", "llc")
check("G-19", "name_aka", "")                                                   # dotted initials: not a marker
check("G-19", "name_latin", "sri aka traders")
assert g.loc["G-20", "is_domain_name"] and g.loc["G-20", "name_has_embedded_phone"]
check("G-20", "name_core", "solutionsrrrcoopers")
check("G-21", "name_latin", "3m 24x7 b2b services")                            # real digit tokens untouched
check("G-23", "name_core", "mitchell premium")
assert set(g.index[g["name_aka"] != ""]) == {"G-14", "G-15"}
assert set(g.index[g["addr_is_multi_location"]]) == {"G-22"}
assert set(g.index[g["name_has_embedded_phone"]]) == {"G-13", "G-17", "G-20"}

print("All golden tests passed.\n")
show_cols = ["name_raw", "name_norm", "name_latin", "name_core", "name_skeleton", "legal_suffix", "name_aka",
             "postal_code", "street_number", "country_norm", "name_script"]
with pd.option_context("display.max_colwidth", 40, "display.width", 250):
    print(g[show_cols].to_string())

_pairs = [("Shree Balaji Traders", "श्री बालाजी ट्रेडर्स"), ("Ganesh Motors", "गणेश मोटर्स"),
          ("City Hardware", "सिटी हार्डवेयर"), ("Metro Cash Carry", "मेट्रो कैश कैरी"),
          ("Delhi Electricals", "दिल्ली इलेक्ट्रिकल्स"), ("Reliance Digital", "रिलायंस डिजिटल"),
          ("Ashok Enterprises", "अशोक इंटरप्राइजेज"), ("Murugan Stores", "முருகன் ஸ்டோர்ஸ்")]
_pdf = pd.DataFrame([(f"P{i}a", a, "", "India") for i, (a, b) in enumerate(_pairs)] +
                    [(f"P{i}b", b, "", "India") for i, (a, b) in enumerate(_pairs)], columns=SOURCE_COLUMNS)
_pp = prepare_chunk(_pdf).set_index("entity_id")
rows = []
for i, (a, b) in enumerate(_pairs):
    la, lb = _pp.loc[f"P{i}a"], _pp.loc[f"P{i}b"]
    rows.append({"latin_side": la["name_latin"], "native->latin": lb["name_latin"],
                 "skeleton_a": la["name_skeleton"], "skeleton_b": lb["name_skeleton"],
                 "latin_equal": la["name_latin"] == lb["name_latin"],
                 "skeleton_equal": la["name_skeleton"] == lb["name_skeleton"]})
print("\nCross-script pairs:")
print(pd.DataFrame(rows).to_string())

## Layer D — country audit + TSV parser check

Collects every raw `country` value in all six files, maps it through `normalize_country`, and
records the map. It also compares parsed row counts with raw line counts: a mismatch means a
quote character is merging rows under pandas' default quoting — switch
`BLOCK1["tsv_quoting"]` to `csv.QUOTE_NONE` and re-run.

In [ ]:
# Layer D: country audit + parser row-count check
t0 = time.time()

country_counts, RAW_ROW_COUNTS, LINE_COUNTS = {}, {}, {}
for (split, source), path in INPUT_FILES.items():
    key = ds_key(split, source)
    vc, rows = None, 0
    for chunk in iter_tsv(path, usecols=["country"], chunksize=1_000_000):
        rows += len(chunk)
        c = chunk["country"].value_counts()
        vc = c if vc is None else vc.add(c, fill_value=0)
    country_counts[key] = {str(k): int(v) for k, v in vc.items()}
    RAW_ROW_COUNTS[key] = rows
    LINE_COUNTS[key] = count_data_lines(path)
RAW_ROW_COUNTS["train_ground_truth"] = sum(len(c) for c in iter_tsv(GT_FILE, chunksize=1_000_000))
LINE_COUNTS["train_ground_truth"] = count_data_lines(GT_FILE)

raw_values = sorted({v for d in country_counts.values() for v in d})
COUNTRY_MAP = {raw: normalize_country(raw) for raw in raw_values}
norm_by_split = {s: sorted({COUNTRY_MAP[v] for k, d in country_counts.items() if k.startswith(s) for v in d})
                 for s in ["train", "test"]}
collapsed = {}
for raw, norm in COUNTRY_MAP.items():
    collapsed.setdefault(norm, []).append(raw)

parser_check = {k: {"parsed_rows": RAW_ROW_COUNTS[k], "raw_lines": LINE_COUNTS[k],
                    "ok": RAW_ROW_COUNTS[k] == LINE_COUNTS[k]} for k in RAW_ROW_COUNTS}
country_report = {
    "raw_value_counts": country_counts,
    "raw_to_norm": COUNTRY_MAP,
    "norm_labels_train": norm_by_split["train"],
    "norm_labels_test": norm_by_split["test"],
    "test_only_labels": sorted(set(norm_by_split["test"]) - set(norm_by_split["train"])),
    "raw_variants_per_norm": {k: v for k, v in collapsed.items() if len(v) > 1},
    "parser_check": parser_check,
}
save_json(country_report, VOCAB_ROOT / "country_map.json")
TIMINGS["D_country_audit"] = round(time.time() - t0, 1)

print("Raw -> norm:", COUNTRY_MAP)
print("Train labels:", norm_by_split["train"], "| Test labels:", norm_by_split["test"],
      "| test-only:", country_report["test_only_labels"])
for k, v in parser_check.items():
    flag = "OK " if v["ok"] else "MISMATCH"
    print(f"  [{flag}] {k:20s} parsed={v['parsed_rows']:>10,d}  lines={v['raw_lines']:>10,d}")
if not all(v["ok"] for v in parser_check.values()):
    print("\n!! Row-count mismatch: embedded quotes/newlines are likely. Try BLOCK1['tsv_quoting'] = csv.QUOTE_NONE.")

## Layer E — ground-truth alignment & integrity

Explodes the GT lists into pairs and removes anything that can't be a valid label (duplicates,
wrong prefixes, IDs that don't exist in S2/S3, S1 IDs missing from Source 1). If a record is
claimed by more than one S1 entity, those S1 entities are merged into one **entity group**
(union-find) so they always receive the same role — otherwise that record would leak across
roles. Outputs used later: `pairs_clean`, `MATCH_LOOKUP` (record → S1), `S1_GROUP` (S1 → group).

In [ ]:
# Layer E: ground-truth alignment & integrity
t0 = time.time()

gt = pd.read_csv(GT_FILE, **READ_KW)
assert {"source1_entity_id", "matched_entity_ids"} <= set(gt.columns), gt.columns.tolist()
gt["source1_entity_id"] = gt["source1_entity_id"].str.strip()
gt["matched_entity_ids"] = gt["matched_entity_ids"].fillna("")

s1_ids = read_tsv_column(INPUT_FILES[("train", "S1")], ["entity_id"])["entity_id"].str.strip()
s2_ids = read_tsv_column(INPUT_FILES[("train", "S2")], ["entity_id"])["entity_id"].str.strip()
s3_ids = read_tsv_column(INPUT_FILES[("train", "S3")], ["entity_id"])["entity_id"].str.strip()
s1_index = pd.Index(s1_ids)
pool_index = pd.Index(pd.concat([s2_ids, s3_ids], ignore_index=True))

integrity = {
    "gt_rows": int(len(gt)),
    "gt_duplicate_s1_rows": int(gt["source1_entity_id"].duplicated().sum()),
    "source1_rows": int(len(s1_ids)),
    "source1_duplicate_ids": int(s1_ids.duplicated().sum()),
    "source2_duplicate_ids": int(s2_ids.duplicated().sum()),
    "source3_duplicate_ids": int(s3_ids.duplicated().sum()),
}
if integrity["gt_duplicate_s1_rows"]:
    gt = (gt.groupby("source1_entity_id", sort=False)["matched_entity_ids"]
            .agg(lambda x: ",".join(v for v in x if v)).reset_index())

gt_index = pd.Index(gt["source1_entity_id"])
integrity["source1_ids_missing_from_gt"] = int((~s1_index.isin(gt_index)).sum())
integrity["gt_s1_ids_not_in_source1"] = int((~gt_index.isin(s1_index)).sum())
integrity["singleton_rows"] = int((gt["matched_entity_ids"].str.strip() == "").sum())

ex = gt.assign(matched_entity_id=gt["matched_entity_ids"].str.split(",")).explode("matched_entity_id")
ex["matched_entity_id"] = ex["matched_entity_id"].fillna("").str.strip()
pairs = ex.loc[ex["matched_entity_id"] != "", ["source1_entity_id", "matched_entity_id"]]
integrity["pairs_raw"] = int(len(pairs))

dup = pairs.duplicated()
bad_prefix = ~pairs["matched_entity_id"].str[:3].isin(["S2-", "S3-"])
dangling = ~bad_prefix & ~pairs["matched_entity_id"].isin(pool_index)
unknown_s1 = ~pairs["source1_entity_id"].isin(s1_index)
integrity.update({
    "pairs_duplicate": int(dup.sum()),
    "pairs_bad_prefix": int(bad_prefix.sum()),
    "pairs_dangling": int(dangling.sum()),
    "pairs_unknown_s1": int(unknown_s1.sum()),
})
integrity["examples"] = {
    "bad_prefix": pairs[bad_prefix].head(10).values.tolist(),
    "dangling": pairs[dangling].head(10).values.tolist(),
    "unknown_s1": pairs[unknown_s1].head(10).values.tolist(),
}
pairs_clean = (pairs[~dup & ~bad_prefix & ~dangling & ~unknown_s1]
               .sort_values(["matched_entity_id", "source1_entity_id"]).reset_index(drop=True))
pairs_clean["matched_source"] = pairs_clean["matched_entity_id"].str[:2]

claims = pairs_clean.groupby("matched_entity_id")["source1_entity_id"].nunique()
multi = claims[claims > 1]
integrity["records_claimed_by_multiple_s1"] = int(len(multi))
integrity["pairs_clean"] = int(len(pairs_clean))
integrity["pairs_clean_by_source"] = {k: int(v) for k, v in pairs_clean["matched_source"].value_counts().items()}

# union-find over S1 entities that share a claimed record
S1_GROUP = {}
if len(multi):
    parent = {}
    def _find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for _, grp in pairs_clean[pairs_clean["matched_entity_id"].isin(multi.index)].groupby("matched_entity_id"):
        members = sorted(grp["source1_entity_id"].unique())
        for m in members[1:]:
            ra, rb = _find(members[0]), _find(m)
            if ra != rb:
                parent[max(ra, rb)] = min(ra, rb)
    S1_GROUP = {s: _find(s) for s in parent}
integrity["s1_entities_in_merged_groups"] = len(S1_GROUP)
integrity["merged_groups"] = len(set(S1_GROUP.values()))

# record -> S1 (first claim in sorted order; positive_pairs stays authoritative for multi-claims)
MATCH_LOOKUP = pairs_clean.drop_duplicates("matched_entity_id").set_index("matched_entity_id")["source1_entity_id"]

save_json(integrity, LABELS_ROOT / "gt_integrity.json")
del ex, pairs, s2_ids, s3_ids
TIMINGS["E_gt_alignment"] = round(time.time() - t0, 1)
print(json.dumps({k: v for k, v in integrity.items() if k != "examples"}, indent=2))

## Layer F — entity profiling for stratification

A light pass (names + country only) that computes, per S1 entity: normalized country, script
group of the S1 name, number of matches by source, and whether any of its true matches is
cross-script. Uses exactly the same normalization functions as Layer H, so the flags agree.

In [ ]:
# Layer F: entity profiling (light pass)
t0 = time.time()

parts = []
for chunk in iter_tsv(INPUT_FILES[("train", "S1")], usecols=["entity_id", "business_name", "country"]):
    norm, _, _, _ = normalize_name(chunk["business_name"])
    sc, mx = detect_scripts(norm)
    parts.append(pd.DataFrame({
        "source1_entity_id": chunk["entity_id"].str.strip().values,
        "country_norm": chunk["country"].map(normalize_country).values,
        "s1_name_script": sc.values,
        "s1_name_mixed": mx.values,
    }))
s1_prof = pd.concat(parts, ignore_index=True)
s1_prof = s1_prof.drop_duplicates("source1_entity_id").reset_index(drop=True)

matched_index = pd.Index(MATCH_LOOKUP.index)
mparts = []
for source in ["S2", "S3"]:
    for chunk in iter_tsv(INPUT_FILES[("train", source)], usecols=["entity_id", "business_name"]):
        ids = chunk["entity_id"].str.strip()
        keep = ids.isin(matched_index)
        if not keep.any():
            continue
        norm, _, _, _ = normalize_name(chunk.loc[keep, "business_name"])
        sc, _ = detect_scripts(norm)
        mparts.append(pd.DataFrame({"matched_entity_id": ids[keep].values, "match_name_script": sc.values}))
match_scripts = (pd.concat(mparts, ignore_index=True) if mparts
                 else pd.DataFrame(columns=["matched_entity_id", "match_name_script"]))

ps = (pairs_clean.merge(match_scripts, on="matched_entity_id", how="left")
                 .merge(s1_prof[["source1_entity_id", "s1_name_script"]], on="source1_entity_id", how="left"))
ps["match_name_script"] = ps["match_name_script"].fillna("empty")
ps["cross_script"] = cross_script_flags(ps["s1_name_script"].fillna("empty"), ps["match_name_script"])

ps["_is_s2"] = ps["matched_source"] == "S2"
ps["_is_s3"] = ps["matched_source"] == "S3"
agg = ps.groupby("source1_entity_id").agg(
    n_matches=("matched_entity_id", "size"),
    n_s2=("_is_s2", "sum"),
    n_s3=("_is_s3", "sum"),
    has_cross_script=("cross_script", "max"),
)
s1_prof = s1_prof.merge(agg, left_on="source1_entity_id", right_index=True, how="left")
for c in ["n_matches", "n_s2", "n_s3"]:
    s1_prof[c] = s1_prof[c].fillna(0).astype("int32")
s1_prof["has_cross_script"] = s1_prof["has_cross_script"].fillna(False).astype(bool)
s1_prof["is_singleton"] = s1_prof["n_matches"] == 0
s1_prof["script_group"] = [script_group(s, m) for s, m in zip(s1_prof["s1_name_script"], s1_prof["s1_name_mixed"])]

edges = BLOCK1["match_bucket_edges"]
bucket_labels = [str(lo) if hi - lo == 1 else f"{lo}-{hi - 1}" for lo, hi in zip(edges[:-1], edges[1:])] + [f"{edges[-1]}+"]
s1_prof["match_bucket"] = pd.cut(s1_prof["n_matches"], bins=edges + [np.inf], right=False,
                                 labels=bucket_labels).astype(str)
s1_prof["stratum"] = (s1_prof["country_norm"] + "|" + s1_prof["script_group"] + "|" +
                      s1_prof["match_bucket"] + "|xs=" + s1_prof["has_cross_script"].astype(int).astype(str))

TIMINGS["F_profiling"] = round(time.time() - t0, 1)
print(f"S1 entities: {len(s1_prof):,} | strata: {s1_prof['stratum'].nunique()} | "
      f"singletons: {s1_prof['is_singleton'].mean():.2%} | with cross-script match: {s1_prof['has_cross_script'].mean():.2%}")
print(f"True pairs profiled: {len(ps):,} | cross-script pairs: {ps['cross_script'].mean():.2%}")
print(s1_prof["script_group"].value_counts().to_string())

## Layer G — entity-level role assignment

Within each stratum (country × script group × match-count bucket × has-cross-script), entities
are shuffled with a fixed seed and dealt into roles in proportion to `BLOCK1["roles"]`.
Merged entity groups (Layer E) are assigned as one unit. Matched S2/S3 records inherit their S1's
role in Layer H, so **no entity is ever split across roles**.

Why six roles: the CatBoost model uses the cross-encoder's logit as a feature, so it must train on
entities the cross-encoder never saw (CE-* vs TREE-*); the threshold is chosen on data neither
model tuned on (CALIBRATION); and HOLDOUT is scored once, at the end.

In [ ]:
# Layer G: entity-level role assignment
t0 = time.time()

prof = s1_prof.sort_values("source1_entity_id").reset_index(drop=True)
prof["group_id"] = prof["source1_entity_id"].map(S1_GROUP).fillna(prof["source1_entity_id"])
reps = prof[prof["group_id"] == prof["source1_entity_id"]].copy()

rng = np.random.default_rng(CONFIG["seed"])
reps["_u"] = rng.random(len(reps))
reps = reps.sort_values(["stratum", "_u"])
reps["_rank"] = reps.groupby("stratum").cumcount()
reps["_n"] = reps.groupby("stratum")["stratum"].transform("size")
cum = np.cumsum([BLOCK1["roles"][r] for r in ROLE_NAMES])
cum[-1] = 1.0
pos = ((reps["_rank"] + 0.5) / reps["_n"]).to_numpy()
reps["entity_role"] = np.array(ROLE_NAMES)[np.searchsorted(cum, pos, side="right")]

prof["entity_role"] = prof["group_id"].map(reps.set_index("source1_entity_id")["entity_role"])
assert prof["entity_role"].notna().all()
ROLE_LOOKUP = prof.set_index("source1_entity_id")["entity_role"]

target = pd.Series(BLOCK1["roles"])
balance = {"overall_share": prof["entity_role"].value_counts(normalize=True).round(4).to_dict()}
for col in ["country_norm", "script_group", "match_bucket", "has_cross_script"]:
    tab = pd.crosstab(prof[col], prof["entity_role"], normalize="index").reindex(columns=ROLE_NAMES, fill_value=0)
    balance[col] = {
        "max_abs_deviation_from_target": round(float((tab - target).abs().max().max()), 4),
        "share_table": tab.round(4).to_dict(orient="index"),
        "counts": prof[col].value_counts().to_dict(),
    }

roles_out = prof[["source1_entity_id", "entity_role", "group_id", "stratum", "country_norm", "s1_name_script",
                  "script_group", "n_matches", "n_s2", "n_s3", "is_singleton", "has_cross_script", "match_bucket"]]
roles_out.to_parquet(LABELS_ROOT / "entity_roles.parquet", index=False, compression="zstd")
save_json(balance, B1_REPORT_ROOT / "role_balance.json")
TIMINGS["G_roles"] = round(time.time() - t0, 1)

summary = prof.groupby("entity_role").agg(
    entities=("source1_entity_id", "size"), true_pairs=("n_matches", "sum"),
    singleton_pct=("is_singleton", "mean"), cross_script_pct=("has_cross_script", "mean"),
).reindex(ROLE_NAMES)
summary["share"] = summary["entities"] / summary["entities"].sum()
print(summary.to_string(float_format=lambda x: f"{x:.4f}"))
print("\nMax deviation from target share:",
      {k: v["max_abs_deviation_from_target"] for k, v in balance.items() if isinstance(v, dict) and "max_abs_deviation_from_target" in v})

## Layer H — full record preparation (chunked, multi-core, resumable)

For each of the six input files: read `chunk_size` rows at a time, normalize them in a worker pool,
attach `source / split / cluster_id / entity_role` in the main process (the big lookups never
leave it), and write `part-NNNNN.parquet` plus a small `_stats_part-NNNNN.json` sidecar.

**Resume:** shards that already exist are skipped, so after an interruption you simply re-run the
notebook from the top. Each dataset folder holds `_CONFIG_HASH`; if you edit Layer B and the
hash changes, this layer stops rather than mix old and new shards — bump `BLOCK1["version"]` or
set `overwrite=True`. `_SUCCESS.json` marks a completed dataset.

In [ ]:
# Layer H: full record preparation
import multiprocessing as mp

def dataset_dir(split, source):
    return RECORDS_ROOT / ds_key(split, source)

def record_files(split, source):
    return sorted(dataset_dir(split, source).glob("part-*.parquet"))

def read_records(split, source, columns=None, row_filter=None):
    """Read prepared records (optionally a column subset / row filter applied per shard)."""
    frames = []
    for f in record_files(split, source):
        df = pq.read_table(f, columns=columns).to_pandas()
        if row_filter is not None:
            df = df[row_filter(df)]
        frames.append(df)
    if not frames:
        return pd.DataFrame(columns=columns or RECORD_COLUMNS)
    return pd.concat(frames, ignore_index=True)

def _finalize(prepared, split, source):
    prepared["source"] = source
    prepared["split"] = split
    if split == "train" and source == "S1":
        cluster = prepared["entity_id"]
        role = cluster.map(ROLE_LOOKUP).fillna("UNASSIGNED")
    elif split == "train":
        cluster = prepared["entity_id"].map(MATCH_LOOKUP).fillna("")
        role = cluster.map(ROLE_LOOKUP).where(cluster != "", "POOL").fillna("UNASSIGNED")
    else:
        cluster = pd.Series("", index=prepared.index)
        role = pd.Series("TEST", index=prepared.index)
    prepared["cluster_id"] = cluster.values
    prepared["entity_role"] = role.values
    return prepared[RECORD_COLUMNS]

def shard_stats(df):
    return {
        "rows": int(len(df)),
        "name_missing": int(df["name_missing"].sum()),
        "addr_missing": int(df["addr_missing"].sum()),
        "name_romanized": int(df["name_was_romanized"].sum()),
        "addr_romanized": int(df["addr_was_romanized"].sum()),
        "name_mixed_script": int(df["name_is_mixed_script"].sum()),
        "name_latin_nonascii_left": int((~df["name_latin"].map(str.isascii)).sum()),
        "addr_latin_nonascii_left": int((~df["addr_latin"].map(str.isascii)).sum()),
        "name_emptied_by_romanization": int(((df["name_norm"] != "") & (df["name_latin"] == "")).sum()),
        "skeleton_empty_nonempty_name": int(((df["name_norm"] != "") & (df["name_skeleton"] == "")).sum()),
        "postal_present": int((df["postal_code"] != "").sum()),
        "street_number_present": int((df["street_number"] != "").sum()),
        "landmark": int(df["has_landmark"].sum()),
        "domain_name": int(df["is_domain_name"].sum()),
        "legal_suffix_present": int((df["legal_suffix"] != "").sum()),
        "name_aka_present": int((df["name_aka"] != "").sum()),
        "name_embedded_phone": int(df["name_has_embedded_phone"].sum()),
        "addr_embedded_phone": int(df["addr_has_embedded_phone"].sum()),
        "addr_multi_location": int(df["addr_is_multi_location"].sum()),
        "cluster_assigned": int((df["cluster_id"] != "").sum()),
        "empty_entity_id": int((df["entity_id"] == "").sum()),
        "nan_cells": int(df.isna().sum().sum()),
        "name_script": {k: int(v) for k, v in df["name_script"].value_counts().items()},
        "addr_script": {k: int(v) for k, v in df["addr_script"].value_counts().items()},
        "country_norm": {k: int(v) for k, v in df["country_norm"].value_counts().items()},
        "entity_role": {k: int(v) for k, v in df["entity_role"].value_counts().items()},
        "legal_suffix": {k: int(v) for k, v in df.loc[df["legal_suffix"] != "", "legal_suffix"].value_counts().items()},
    }

def merge_stats(stats_list):
    total = {}
    for s in stats_list:
        for k, v in s.items():
            if isinstance(v, dict):
                d = total.setdefault(k, {})
                for kk, vv in v.items():
                    d[kk] = d.get(kk, 0) + vv
            else:
                total[k] = total.get(k, 0) + v
    return total

LATIN_ONLY_COLUMNS = ["name_latin", "addr_latin", "name_core", "name_skeleton", "name_aka", "legal_suffix"]

def assert_latin_only(df, context=""):
    """Sanity gate run on every chunk BEFORE it is written to parquet: every column later
    blocks treat as the romanized / translated matching key must actually be pure ASCII --
    a correctly-translated legal suffix, a correctly-romanized name, nothing left over from
    a script anyascii (or the revision 3/4 native-vocab canonicalization) couldn't fully
    convert. Catches anyascii edge cases with no mapping for some character, a native-vocab
    substitution that left a stray token behind, or any future normalization change that lets
    non-ASCII text leak through. Deliberately does NOT check name_norm / addr_norm -- those
    are supposed to keep native script (Block 4 needs it); this only gates the *_latin /
    name_core / name_skeleton / name_aka / legal_suffix columns Block 2+ actually key on."""
    problems = []
    for col in LATIN_ONLY_COLUMNS:
        if col not in df.columns:
            continue
        bad = ~df[col].map(str.isascii)
        if bad.any():
            cols_to_show = [c for c in ("entity_id", "name_script", "addr_script", col) if c in df.columns]
            problems.append((col, int(bad.sum()), df.loc[bad, cols_to_show].head(5)))
    if problems:
        lines = [f"LATIN-ONLY SANITY CHECK FAILED{' (' + context + ')' if context else ''}: "
                 f"non-ASCII characters found in column(s) that later blocks treat as pure-Latin "
                 f"matching keys. This means some native-script text is reaching Block 2+ untranslated."]
        for col, n_bad, sample in problems:
            lines.append(f"\n  {col}: {n_bad:,} affected row(s), examples:\n{sample.to_string(index=False)}")
        raise AssertionError("".join(lines))

def _write_shard(df, path):
    tmp = path.with_name("." + path.name + ".tmp")
    df = df.astype({c: "bool" for c, (t, _, _) in RECORD_SCHEMA.items() if t == "bool"})
    pq.write_table(pa.Table.from_pandas(df, preserve_index=False), tmp, compression="zstd")
    os.replace(tmp, path)

def _worker(args):
    idx, chunk = args
    return idx, prepare_chunk(chunk)

def prepare_dataset(split, source, pool):
    path = INPUT_FILES[(split, source)]
    out_dir = dataset_dir(split, source)
    success, hash_file = out_dir / "_SUCCESS.json", out_dir / "_CONFIG_HASH"
    if BLOCK1["overwrite"] and out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    if hash_file.exists() and hash_file.read_text().strip() != CONFIG_HASH:
        raise RuntimeError(f"{out_dir} was built with a different CONFIG_HASH. Bump BLOCK1['version'] "
                           f"or set BLOCK1['overwrite']=True.")
    hash_file.write_text(CONFIG_HASH)
    if success.exists():
        print(f"  {ds_key(split, source)}: complete, skipping")
        return json.loads(success.read_text())["stats"]

    t_start, done_rows, batch = time.time(), 0, []
    batch_size = max(1, BLOCK1["n_workers"] * 2)

    def _flush(batch):
        nonlocal done_rows
        results = pool.map(_worker, batch) if pool is not None else [_worker(b) for b in batch]
        for idx, prepared in results:
            final = _finalize(prepared, split, source)
            assert_latin_only(final, context=f"{ds_key(split, source)} part-{idx:05d}")
            _write_shard(final, out_dir / f"part-{idx:05d}.parquet")
            save_json(shard_stats(final), out_dir / f"_stats_part-{idx:05d}.json")
            done_rows += len(final)
        rate = done_rows / max(1e-9, time.time() - t_start)
        print(f"    {ds_key(split, source)}: {done_rows:,} new rows written ({rate:,.0f} rows/s)")

    for idx, chunk in enumerate(iter_tsv(path, usecols=SOURCE_COLUMNS)):
        shard = out_dir / f"part-{idx:05d}.parquet"
        if shard.exists() and (out_dir / f"_stats_part-{idx:05d}.json").exists():
            continue
        batch.append((idx, chunk))
        if len(batch) >= batch_size:
            _flush(batch)
            batch = []
    if batch:
        _flush(batch)

    stats = merge_stats([json.loads(p.read_text()) for p in sorted(out_dir.glob("_stats_part-*.json"))])
    save_json({"config_hash": CONFIG_HASH, "finished_at": datetime.now().isoformat(),
               "shards": len(record_files(split, source)), "stats": stats}, success)
    return stats

t0 = time.time()
DATASET_STATS = {}
pool = mp.get_context("fork").Pool(BLOCK1["n_workers"]) if BLOCK1["n_workers"] > 1 else None
try:
    for (split, source) in INPUT_FILES:
        ts = time.time()
        print(f"Preparing {ds_key(split, source)} ...")
        DATASET_STATS[ds_key(split, source)] = prepare_dataset(split, source, pool)
        TIMINGS[f"H_{ds_key(split, source)}"] = round(time.time() - ts, 1)
finally:
    if pool is not None:
        pool.close()
        pool.join()
TIMINGS["H_total"] = round(time.time() - t0, 1)

for k, s in DATASET_STATS.items():
    print(f"{k:10s} rows={s['rows']:>10,d}  romanized(name)={s['name_romanized']:>8,d}  "
          f"postal={s['postal_present'] / max(1, s['rows']):6.2%}  cluster={s['cluster_assigned']:>9,d}  "
          f"roles={s['entity_role']}")

# Every shard already passed assert_latin_only() before it was written (see prepare_dataset /
# _flush above) -- reaching this line at all means every one of these rows passed. This is
# just the explicit, printed confirmation of that gate, not a re-check of the data.
_total_rows = sum(s["rows"] for s in DATASET_STATS.values())
print(f"\nLatin-only sanity check: PASSED for all {_total_rows:,} rows across every shard -- "
      f"name_latin / addr_latin / name_core / name_skeleton / name_aka / legal_suffix are pure "
      f"ASCII everywhere. (Any violation would have raised AssertionError and stopped Layer H "
      f"before that shard was written -- see assert_latin_only.)")

## Layer I — positive-pair label table

`labels/positive_pairs.parquet` is the authoritative label source for every later block:
one row per true (S1, S2/S3) pair with role, country, both scripts and the cross-script flag.

In [ ]:
# Layer I: positive-pair label table
t0 = time.time()

s1_attr = read_records("train", "S1", ["entity_id", "country_norm", "name_script", "entity_role"]).rename(
    columns={"entity_id": "source1_entity_id", "name_script": "s1_name_script", "entity_role": "entity_role"})
matched_index = pd.Index(pairs_clean["matched_entity_id"].unique())
m_attr = pd.concat([
    read_records("train", src, ["entity_id", "name_script", "entity_role", "cluster_id"],
                 row_filter=lambda d: d["entity_id"].isin(matched_index))
    for src in ["S2", "S3"]], ignore_index=True).rename(
    columns={"entity_id": "matched_entity_id", "name_script": "match_name_script",
             "entity_role": "match_role", "cluster_id": "match_cluster_id"})

pp = (pairs_clean.merge(s1_attr, on="source1_entity_id", how="left")
                 .merge(m_attr, on="matched_entity_id", how="left"))
pp["cross_script"] = cross_script_flags(pp["s1_name_script"].fillna("empty"), pp["match_name_script"].fillna("empty"))

label_checks = {
    "pairs": int(len(pp)),
    "pairs_missing_s1_record": int(pp["s1_name_script"].isna().sum()),
    "pairs_missing_match_record": int(pp["match_name_script"].isna().sum()),
    "pairs_role_mismatch": int((pp["entity_role"] != pp["match_role"]).sum()),
    "pairs_on_multi_claimed_records": int(pp["matched_entity_id"].isin(multi.index).sum()),
    "pairs_cluster_id_disagrees": int(((pp["match_cluster_id"] != pp["source1_entity_id"])
                                       & ~pp["matched_entity_id"].isin(multi.index)).sum()),
    "cross_script_pct": round(100 * float(pp["cross_script"].mean()), 3),
}
positive_pairs = pp[["source1_entity_id", "matched_entity_id", "matched_source", "entity_role",
                     "country_norm", "s1_name_script", "match_name_script", "cross_script"]]
positive_pairs.to_parquet(LABELS_ROOT / "positive_pairs.parquet", index=False, compression="zstd")

xs_by_s1 = positive_pairs.groupby("source1_entity_id")["cross_script"].any()
prof_xs = s1_prof.set_index("source1_entity_id")["has_cross_script"]
label_checks["cross_script_flag_disagreements_vs_layer_F"] = int(
    (prof_xs.reindex(xs_by_s1.index).fillna(False).astype(bool) != xs_by_s1).sum())
TIMINGS["I_positive_pairs"] = round(time.time() - t0, 1)

print(json.dumps(label_checks, indent=2))
print(positive_pairs.groupby("entity_role").agg(pairs=("matched_entity_id", "size"),
                                                cross_script=("cross_script", "mean")).reindex(ROLE_NAMES).to_string())

## Layer J — QA & leakage checks

Every check is recorded as PASS/FAIL in `reports/block1_qa_report.json`. A FAIL doesn't stop the
notebook — read the detail and decide — but Block 2 should not start on a FAIL in the
*leakage* or *row conservation* groups.

In [ ]:
# Layer J: QA & leakage checks
t0 = time.time()
checks = []

def add_check(group, name, passed, detail=None):
    checks.append({"group": group, "check": name, "passed": bool(passed), "detail": detail})

# row conservation + id integrity
for (split, source) in INPUT_FILES:
    k = ds_key(split, source)
    s = DATASET_STATS[k]
    add_check("row_conservation", f"{k}: written rows == parsed rows", s["rows"] == RAW_ROW_COUNTS[k],
              {"written": s["rows"], "parsed": RAW_ROW_COUNTS[k]})
    ids = read_records(split, source, ["entity_id"])["entity_id"]
    add_check("ids", f"{k}: entity_id unique", not ids.duplicated().any(), int(ids.duplicated().sum()))
    add_check("ids", f"{k}: entity_id prefix is {source}-", bool(ids.str.startswith(source + "-").all()),
              int((~ids.str.startswith(source + "-")).sum()))
    add_check("schema", f"{k}: no NaN cells", s["nan_cells"] == 0, s["nan_cells"])
    add_check("romanization", f"{k}: name_latin / addr_latin fully ASCII",
              s["name_latin_nonascii_left"] == 0 and s["addr_latin_nonascii_left"] == 0,
              {"name": s["name_latin_nonascii_left"], "addr": s["addr_latin_nonascii_left"]})
    add_check("romanization", f"{k}: no names emptied by romanization", s["name_emptied_by_romanization"] == 0,
              s["name_emptied_by_romanization"])
    allowed = set(ROLE_NAMES) | ({"POOL"} if split == "train" and source != "S1" else set()) if split == "train" else {"TEST"}
    add_check("roles", f"{k}: roles within {sorted(allowed)}", set(s["entity_role"]) <= allowed, s["entity_role"])

# schema / dtypes on one shard per dataset
for (split, source) in INPUT_FILES:
    files = record_files(split, source)
    if files:
        sch = pq.read_schema(files[0])
        add_check("schema", f"{ds_key(split, source)}: columns match RECORD_SCHEMA",
                  list(sch.names) == RECORD_COLUMNS, [n for n in RECORD_COLUMNS if n not in sch.names])

# leakage
add_check("leakage", "positive pairs never cross roles", label_checks["pairs_role_mismatch"] == 0,
          label_checks["pairs_role_mismatch"])
add_check("leakage", "every train S1 has a role", "UNASSIGNED" not in DATASET_STATS["train_s1"]["entity_role"],
          DATASET_STATS["train_s1"]["entity_role"].get("UNASSIGNED", 0))
add_check("leakage", "merged entity groups share one role",
          bool(prof.groupby("group_id")["entity_role"].nunique().max() == 1))
add_check("labels", "cluster_id agrees with positive_pairs (records claimed once)",
          label_checks["pairs_cluster_id_disagrees"] == 0, label_checks["pairs_cluster_id_disagrees"])
add_check("warning", "no multi-claimed records (else label via positive_pairs, not cluster_id)",
          label_checks["pairs_on_multi_claimed_records"] == 0, label_checks["pairs_on_multi_claimed_records"])
add_check("labels", "every positive pair has both records", label_checks["pairs_missing_s1_record"] == 0
          and label_checks["pairs_missing_match_record"] == 0,
          {"s1": label_checks["pairs_missing_s1_record"], "match": label_checks["pairs_missing_match_record"]})
matched_rows = DATASET_STATS["train_s2"]["cluster_assigned"] + DATASET_STATS["train_s3"]["cluster_assigned"]
add_check("labels", "matched train S2/S3 rows == unique matched ids in GT",
          matched_rows == pairs_clean["matched_entity_id"].nunique(),
          {"records_with_cluster": matched_rows, "gt_unique_matched": int(pairs_clean["matched_entity_id"].nunique())})

# country
test_labels = set().union(*[DATASET_STATS[ds_key("test", s)]["country_norm"] for s in ["S1", "S2", "S3"]])
train_labels = set().union(*[DATASET_STATS[ds_key("train", s)]["country_norm"] for s in ["S1", "S2", "S3"]])
add_check("country", "test rows kept for every country (incl. unseen)", True,
          {"test_labels": sorted(test_labels), "unseen_in_train": sorted(test_labels - train_labels)})

qa_df = pd.DataFrame(checks)
n_fail = int((~qa_df["passed"]).sum())
role_table = (roles_out.groupby("entity_role").agg(entities=("source1_entity_id", "size"),
                                                    singletons=("is_singleton", "sum"),
                                                    true_pairs=("n_matches", "sum"))
              .reindex(ROLE_NAMES).reset_index().to_dict(orient="records"))
qa_report = {
    "run_id": RUN_ID, "config_hash": CONFIG_HASH, "n_checks": len(checks), "n_failed": n_fail,
    "checks": checks, "gt_integrity": {k: v for k, v in integrity.items() if k != "examples"},
    "label_checks": label_checks, "role_table": role_table, "dataset_stats": DATASET_STATS,
    "role_balance_max_deviation": {k: v["max_abs_deviation_from_target"] for k, v in balance.items()
                                   if isinstance(v, dict) and "max_abs_deviation_from_target" in v},
}
save_json(qa_report, B1_REPORT_ROOT / "block1_qa_report.json")
TIMINGS["J_qa"] = round(time.time() - t0, 1)

with pd.option_context("display.max_colwidth", 70, "display.width", 200):
    print(qa_df.assign(status=np.where(qa_df["passed"], "PASS", "FAIL"))[["status", "group", "check"]].to_string(index=False))
print(f"\n{len(checks) - n_fail}/{len(checks)} checks passed.")
if n_fail:
    print(qa_df[~qa_df["passed"]][["check", "detail"]].to_string(index=False))

# examples per script (train S2 first shard + a France sample from test)
ex_cols = ["name_raw", "name_latin", "name_core", "name_skeleton", "legal_suffix", "postal_code", "country_norm"]
sample_src = pq.read_table(record_files("train", "S2")[0]).to_pandas()
print("\nExamples by name script (train S2, first shard):")
for sc, grp in sample_src.groupby("name_script"):
    with pd.option_context("display.max_colwidth", 38, "display.width", 250):
        print(f"\n[{sc}]\n" + grp[ex_cols].head(3).to_string(index=False))
for sc in sorted(test_labels - train_labels):
    unseen_sample = read_records("test", "S1", ex_cols, row_filter=lambda d, sc=sc: d["country_norm"] == sc).head(5)
    with pd.option_context("display.max_colwidth", 38, "display.width", 250):
        print(f"\n[unseen country {sc} — test S1]\n" + unseen_sample.to_string(index=False))

## Layer K — Block 2 readiness bake-off (CE-DEV only)

Same method as the EDA's Layer I, now on the **prepared** fields: for S1 queries from the CE-DEV
role (half with a cross-script match, half without), build a same-country candidate pool from all
train S2+S3 records (random sample up to `bakeoff_pool_cap` + the true matches forced in), fit a
char n-gram TF-IDF per key, retrieve top-K, and measure recall by script agreement.

This tells Block 2 which keys to index and whether a union of two keys pays for its extra
candidates. It is a relative comparison — absolute recall on the full pool will be lower.

In [ ]:
# Layer K: Block 2 readiness bake-off
from sklearn.feature_extraction.text import TfidfVectorizer
t0 = time.time()

role = BLOCK1["bakeoff_role"]
K = BLOCK1["bakeoff_k"]
dev_pairs = positive_pairs[positive_pairs["entity_role"] == role]
half = BLOCK1["bakeoff_queries"] // 2
xs_ids = dev_pairs.loc[dev_pairs["cross_script"], "source1_entity_id"].drop_duplicates()
ss_ids = dev_pairs.loc[~dev_pairs["source1_entity_id"].isin(xs_ids), "source1_entity_id"].drop_duplicates()
q_ids = pd.concat([xs_ids.sample(n=min(half, len(xs_ids)), random_state=CONFIG["seed"]),
                   ss_ids.sample(n=min(half, len(ss_ids)), random_state=CONFIG["seed"])])
eval_pairs = dev_pairs[dev_pairs["source1_entity_id"].isin(q_ids)]
targets = pd.Index(eval_pairs["matched_entity_id"].unique())

KEY_COLS = ["entity_id", "country_norm", "name_norm", "name_latin", "name_core", "name_skeleton", "addr_latin"]
queries = read_records("train", "S1", KEY_COLS, row_filter=lambda d: d["entity_id"].isin(q_ids))

pool_totals = {}
for src in ["S2", "S3"]:
    for c, n in DATASET_STATS[ds_key("train", src)]["country_norm"].items():
        pool_totals[c] = pool_totals.get(c, 0) + n
rng_k = np.random.default_rng(CONFIG["seed"])
pool_parts = []
for src in ["S2", "S3"]:
    for f in record_files("train", src):
        d = pq.read_table(f, columns=KEY_COLS).to_pandas()
        p = d["country_norm"].map(lambda c: min(1.0, BLOCK1["bakeoff_pool_cap"] / max(1, pool_totals.get(c, 1))))
        keep = d["entity_id"].isin(targets) | (rng_k.random(len(d)) < p.to_numpy())
        pool_parts.append(d[keep])
pool_df = pd.concat(pool_parts, ignore_index=True).drop_duplicates("entity_id")

KEYS = {
    "name_norm (native, no romanization)": lambda d: d["name_norm"],
    "name_latin": lambda d: d["name_latin"],
    "name_core": lambda d: d["name_core"],
    "name_skeleton": lambda d: d["name_skeleton"],
    "name_latin + addr_latin": lambda d: d["name_latin"] + " " + d["addr_latin"],
}

def _topk(pool_texts, query_texts, k):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1, dtype=np.float32)
    try:
        pm = vec.fit_transform(pool_texts)
    except ValueError:
        return [np.array([], dtype=int)] * len(query_texts)
    qm = vec.transform(query_texts)
    k = min(k, pm.shape[0])
    out = []
    for i in range(0, qm.shape[0], 256):
        sims = (qm[i:i + 256] @ pm.T).toarray()
        top = np.argpartition(-sims, kth=k - 1, axis=1)[:, :k]
        out.extend(list(top))
    return out

retrieved = {key: {} for key in KEYS}
for country, qg in queries.groupby("country_norm"):
    pg = pool_df[pool_df["country_norm"] == country].reset_index(drop=True)
    if pg.empty:
        continue
    pool_ids = pg["entity_id"].to_numpy()
    for key, fn in KEYS.items():
        tops = _topk(fn(pg).tolist(), fn(qg).tolist(), K)
        for qid, top in zip(qg["entity_id"], tops):
            retrieved[key][qid] = set(pool_ids[top])
retrieved["UNION name_latin | name_skeleton"] = {
    q: retrieved["name_latin"].get(q, set()) | retrieved["name_skeleton"].get(q, set()) for q in q_ids}

rows = []
for key, rmap in retrieved.items():
    hits = [(t in rmap.get(s, set()), xs) for s, t, xs in
            zip(eval_pairs["source1_entity_id"], eval_pairs["matched_entity_id"], eval_pairs["cross_script"])]
    h = pd.DataFrame(hits, columns=["hit", "xs"])
    rows.append({
        "key": key,
        "recall_all": round(100 * h["hit"].mean(), 2),
        "recall_same_script": round(100 * h.loc[~h["xs"], "hit"].mean(), 2) if (~h["xs"]).any() else None,
        "recall_cross_script": round(100 * h.loc[h["xs"], "hit"].mean(), 2) if h["xs"].any() else None,
        "avg_candidates_per_query": round(float(np.mean([len(v) for v in rmap.values()])) if rmap else 0, 1),
    })
bakeoff = pd.DataFrame(rows)
bakeoff_report = {
    "role": role, "k": K, "queries": int(len(q_ids)), "pairs": int(len(eval_pairs)),
    "cross_script_pairs": int(eval_pairs["cross_script"].sum()),
    "pool_size_by_country": pool_df["country_norm"].value_counts().to_dict(),
    "results": rows,
}
save_json(bakeoff_report, B1_REPORT_ROOT / "block2_key_bakeoff.json")
TIMINGS["K_bakeoff"] = round(time.time() - t0, 1)
print(f"{role}: {len(q_ids)} queries, {len(eval_pairs)} true pairs "
      f"({int(eval_pairs['cross_script'].sum())} cross-script), K={K}, pool={len(pool_df):,}")
print(bakeoff.to_string(index=False))

## Layer L — manifest & interface

In [ ]:
# Layer L: manifest

def _file_info(p):
    st = Path(p).stat()
    return {"path": str(p), "bytes": st.st_size, "mtime": datetime.fromtimestamp(st.st_mtime).isoformat()}

manifest = {
    "block": "block1_data_ground_truth",
    "version": BLOCK1["version"],
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(),
    "config_hash": CONFIG_HASH,
    "config": CONFIG,
    "block1_config": {k: v for k, v in BLOCK1.items()},
    "versions": VERSIONS,
    "inputs": {**{ds_key(s, src): _file_info(p) for (s, src), p in INPUT_FILES.items()},
               "train_ground_truth": _file_info(GT_FILE)},
    "raw_row_counts": RAW_ROW_COUNTS,
    "outputs": {
        "records": {ds_key(s, src): {"dir": str(dataset_dir(s, src)), "shards": len(record_files(s, src)),
                                     "rows": DATASET_STATS[ds_key(s, src)]["rows"]} for (s, src) in INPUT_FILES},
        "positive_pairs": str(LABELS_ROOT / "positive_pairs.parquet"),
        "entity_roles": str(LABELS_ROOT / "entity_roles.parquet"),
        "gt_integrity": str(LABELS_ROOT / "gt_integrity.json"),
        "country_map": str(VOCAB_ROOT / "country_map.json"),
        "qa_report": str(B1_REPORT_ROOT / "block1_qa_report.json"),
        "role_balance": str(B1_REPORT_ROOT / "role_balance.json"),
        "bakeoff": str(B1_REPORT_ROOT / "block2_key_bakeoff.json"),
    },
    "record_schema": {c: {"dtype": t, "description": d, "consumers": u} for c, (t, d, u) in RECORD_SCHEMA.items()},
    "roles": BLOCK1["roles"],
    "role_table": role_table,
    "label_semantics": ("positive_pairs.parquet is authoritative. For a candidate (s1, x): label = 1 iff "
                        "(s1, x) in positive_pairs (equivalently cluster_id[x] == s1 when no multi-claims)."),
    "pool_semantics": ("Train S2/S3 records without a match have entity_role='POOL' and may appear as negatives "
                       "for queries of any role; retrieve against the full pool to mirror test conditions."),
    "qa_summary": {"n_checks": qa_report["n_checks"], "n_failed": qa_report["n_failed"]},
    "bakeoff_summary": bakeoff_report["results"],
    "timings_sec": TIMINGS,
}
save_json(manifest, BLOCK1_ROOT / "manifest.json")

print("=" * 72)
print(f"BLOCK 1 COMPLETE — {BLOCK1_ROOT}")
print("=" * 72)
print(f"config_hash {CONFIG_HASH} | QA {qa_report['n_checks'] - qa_report['n_failed']}/{qa_report['n_checks']} passed")
for k, v in manifest["outputs"]["records"].items():
    print(f"  records/{k:10s} {v['rows']:>11,d} rows in {v['shards']} shards")
print(f"  labels/positive_pairs  {label_checks['pairs']:,} pairs ({label_checks['cross_script_pct']}% cross-script)")
print(f"  labels/entity_roles    {len(roles_out):,} S1 entities")
print("\nBake-off (recall@K, %):")
print(bakeoff[["key", "recall_same_script", "recall_cross_script", "avg_candidates_per_query"]].to_string(index=False))
print("\nTimings (s):", TIMINGS)

## How the later blocks read these outputs

```python
import er_eval as ev
ev.set_root(str(WORKSPACE))

s1     = ev.read_records("train", "S1", ["entity_id", "country_norm", "name_latin", "name_skeleton"],
                         roles=["CE-FIT"])                    # per-role queries
pool   = pd.concat([ev.read_records("train", s, [...]) for s in ("S2", "S3")])   # FULL pool, incl. POOL rows
labels = pd.read_parquet(ev.b1_path("labels", "positive_pairs.parquet"))
roles  = pd.read_parquet(ev.b1_path("labels", "entity_roles.parquet"))          # includes singletons
```

**Per block**
- **Block 2:** gate by `country_norm`, retrieve with the fine-tuned bi-encoder on
  `name_norm | addr_norm`; label candidates by joining `positive_pairs`; candidate recall per role is
  Block 2's own metric.
- **Block 3:** features from `name_latin/core/skeleton`, `addr_latin`, `postal_code`, `street_number`,
  `legal_suffix`, `is_domain_name`, `*_missing`, `*_script`, `name_was_romanized`, `country_norm`,
  `name_aka`, `*_has_embedded_phone`, `addr_is_multi_location`.
- **Block 4:** `name_norm` / `addr_norm` (native script) feed the cross-encoder; gradients from
  CE-FIT, checkpoint chosen on CE-DEV.
- **Block 5:** CatBoost on TREE-FIT, early stopping on TREE-DEV.
- **Block 6:** accept threshold on CALIBRATION only; HOLDOUT is read once for the report; then the
  test set.

Next: `02_block2_candidategen_india.ipynb`.